# Notebook 02: Statistical Analysis, Correlation, Network & Progression (v2.1)

**Cohort scope:** `YACHIDA-CRC-2019` (discovery) plus `SINHA_CRC_2016` and
`Kim_adenomas_2020` (external validation) — `utils.DATASETS_ANALYSIS`. The IBD/IBS cohorts
appear only in NB01's exploratory sections. `ERAWIJANTARI-GASTRIC-CANCER-2020` is never used:
53 of its samples are the same Healthy subjects as Yachida's, so it is not independent.

## Sample scope: the primary analysis set

Everything in this notebook runs on the **primary** set, `utils.primary_mask()` —
Healthy + Early_CRC + Advanced_CRC, **n=277** for Yachida. The two groups that are not on
that axis are handled once, here, and never leak downstream:

- **MP** (multiple polypoid adenomas) was dropped at load time in NB01 and does not exist
  in any table this notebook reads.
- **PostSurgery** (HS — cancer resected, currently cancer-free) is split out of the
  matrices in the first section below. The reduced pickle that NB03+ consume therefore
  contains the **primary samples only** under its usual keys, with the PostSurgery rows
  stored separately as `sensitivity_species` / `sensitivity_mtb`. A downstream notebook that
  simply loads `transformed_species` gets the correct sample set by default rather than by
  remembering to filter — the failure mode is absence, not contamination.
  NB01b already quantified how different that group is (its own FDR family).

## Differential abundance is run two ways, and only agreement counts

A rank test and a covariate-adjusted compositional model answer different questions, and the
earlier pipeline reported only the first:

| | Test | Adjusts for confounders | Compositional |
|---|---|---|---|
| `utils.differential_abundance` | Mann-Whitney U + Cliff's δ | no | no |
| `utils.clr_lm_da` | linear model on CLR values | **yes** | **yes** (LinDA bias correction) |

A feature is a **robust DA hit** only when *both* reach q<0.05 **and agree in direction**.
This matters in this cohort specifically: smoking is strongly imbalanced across groups
(median Brinkman index 100 in Healthy vs 196–390 in the CRC groups), so an unadjusted test
cannot distinguish a cancer effect from a smoking effect. Features that pass the rank test
but fail the adjusted model are reported, not hidden — they are candidates explained by
confounding.

**Metabolites** are tested with the same linear model but with `bias_correct=False` and
`already_clr=True`: the metabolite matrix is log10-transformed, not compositional, so the
LinDA mode-subtraction does not apply to it. Only the species model is compositional.

## Steps
- **Primary split + global feature reduction** — top-variance species/metabolites retained
  for this notebook and all of NB03+, saved to `preprocessed_data_reduced_v2.pkl`.
- **Step 6A** — confound / residual diagnostics (the old version's OLS fits all failed
  silently and always reported R²=0.000; fixed via `utils.build_formula`).
- **Step 6** — DA on the fine-grained `Study.Group` scheme (`MeanDiff`, not the old,
  frequently-NaN `Log2FC`).
- **Step 6B** — DA on the grouped `Stage.Group` scheme (Healthy / Early_CRC / Advanced_CRC),
  **both tests**, producing the robust-hit tables NB03 selects its targets from.
- **Step 7** — correlations: species–polyamine, full species–metabolite, partial
  correlations adjusted for the full confounder set (saved for NB09), and **three-cohort**
  replication (Yachida vs Sinha vs Kim) at **genus** level with metabolites matched by
  KEGG ID.
- **Step 8** — networks: species–metabolite co-occurrence, polyamine-centric subnetworks,
  and stage-stratified hub networks.
- **Step 10** — progression: ordinal Spearman against CRC stage (10A) and binary
  early-vs-late DA (10B).

There is **no ML step in this notebook.** The old version's intro advertised a "Step 9: ML
Pipeline" for which no code existed anywhere in it. All ML lives in `03_ml_benchmarking.ipynb`,
which reads this notebook's `preprocessed_data_reduced_v2.pkl`.

In [ ]:
import sys
sys.path.insert(0, '.')

import os
import pickle
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import ols as sm_ols
from scipy.stats import probplot

import utils
from utils import (
    RESULTS_DIR,
    DATASETS_ANALYSIS, DATASET_DISCOVERY, DATASETS_VALIDATION,
    POLYAMINE_DATASETS_ANALYSIS, POLYAMINE_KEGG,
    COMPARISONS, COMPARISONS_GROUP,
    CRC_STAGE_ORDER, CRC_STAGE_NUMERIC, CRC_PROGRESSION_COMPARISONS,
    STAGE_GROUP_MAP, STAGE_GROUP_ORDER,
    YACHIDA_COMPLETE_CONFOUNDERS,
    MAX_SPECIES_REDUCED, MAX_MTB_REDUCED, FDR_THRESHOLD,
    differential_abundance, clr_lm_da,
    compute_correlations, find_consistent_correlations,
    volcano_plot, plot_correlation_heatmap,
    progression_correlation_analysis, plot_stage_trend,
    build_formula, primary_mask, sensitivity_mask, apply_mask,
)

for sub in ['figures/differential', 'figures/correlation', 'figures/network',
            'figures/progression', 'tables', 'intermediate']:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded.')
print(f'Discovery : {DATASET_DISCOVERY}')
print(f'Validation: {DATASETS_VALIDATION}')

In [ ]:
# Load preprocessed data from Notebook 01
with open(RESULTS_DIR / 'intermediate' / 'preprocessed_data_v2.pkl', 'rb') as f:
    preprocessed = pickle.load(f)

transformed_species = {ds: preprocessed['transformed_species'][ds] for ds in DATASETS_ANALYSIS}
transformed_mtb     = {ds: preprocessed['transformed_mtb'][ds]     for ds in DATASETS_ANALYSIS}
harmonized_meta     = {ds: preprocessed['harmonized_meta'][ds]     for ds in DATASETS_ANALYSIS}
polyamine_columns   = {ds: preprocessed['polyamine_columns'].get(ds, {})
                       for ds in DATASETS_ANALYSIS}
polyamine_proxies   = {ds: preprocessed.get('polyamine_proxies', {}).get(ds, {})
                       for ds in DATASETS_ANALYSIS}
# mtb_kegg_map: {dataset: {mtb_column_name: kegg_id}} -- built in NB01 from each
# dataset's own mtb.map file. Needed below for KEGG-ID-based cross-cohort matching
# (Step 7.4), since the three cohorts use different metabolomics naming conventions.
mtb_kegg_map  = {ds: preprocessed['mtb_kegg_map'].get(ds, {}) for ds in DATASETS_ANALYSIS}
MTB_PRESCALED = preprocessed.get('mtb_prescaled', {})

# MP must be gone already (dropped at load in NB01).
assert not harmonized_meta[DATASET_DISCOVERY]['Study.Group'].eq('MP').any(), \
    'MP must have been excluded in NB01'

# ---------------------------------------------------------------------------
# PRIMARY SPLIT -- done once, here, for every cohort.
# The sensitivity group (PostSurgery / HS) is removed from the working matrices
# and kept aside. Everything after this cell operates on the primary set only.
# ---------------------------------------------------------------------------
sensitivity_species, sensitivity_mtb, sensitivity_meta = {}, {}, {}

for ds in DATASETS_ANALYSIS:
    meta = harmonized_meta[ds]
    pm, sm = primary_mask(meta), sensitivity_mask(meta)

    if sm.any():
        s_spc, s_mtb = apply_mask(sm, transformed_species[ds], transformed_mtb[ds], meta_df=meta)
        sensitivity_species[ds] = s_spc
        sensitivity_mtb[ds]     = s_mtb
        sensitivity_meta[ds]    = meta[sm].reset_index(drop=True)

    p_spc, p_mtb = apply_mask(pm, transformed_species[ds], transformed_mtb[ds], meta_df=meta)
    transformed_species[ds] = p_spc
    transformed_mtb[ds]     = p_mtb
    harmonized_meta[ds]     = meta[pm].reset_index(drop=True)

    print(f'{ds}: primary n={int(pm.sum())}'
          + (f', held aside as sensitivity n={int(sm.sum())}' if sm.any() else ''))

# Hard guard: no non-primary group may survive in the working tables.
for ds in DATASETS_ANALYSIS:
    bad = set(harmonized_meta[ds]['Stage.Group'].dropna()) - set(STAGE_GROUP_ORDER)
    assert not bad, f'{ds}: non-primary group(s) still present: {bad}'
    for name, tbl in (('species', transformed_species[ds]), ('mtb', transformed_mtb[ds])):
        assert set(tbl['Sample']) == set(harmonized_meta[ds]['Sample']), \
            f'{ds}: {name} samples do not match the primary metadata'

print('\nPreprocessed data loaded; primary analysis set in force.')
for ds in DATASETS_ANALYSIS:
    n_spc  = len(transformed_species[ds].columns) - 1
    n_mtb  = len(transformed_mtb[ds].columns) - 1
    n_pa   = len(polyamine_columns[ds])
    n_px   = len(polyamine_proxies[ds])
    n_kegg = len(mtb_kegg_map[ds])
    print(f'  {ds}: {len(transformed_species[ds])} samples, {n_spc} species, '
          f'{n_mtb} metabolites, {n_pa} polyamines (+{n_px} proxy), {n_kegg} KEGG mappings')

---
## Primary split + Global Feature Reduction

Applied immediately after loading. All subsequent cells here (DA, correlations, hub
networks, progression) and NB03+ (ML) use this reduced feature set. Sizes come from the
shared constants `utils.MAX_SPECIES_REDUCED` (500) and `utils.MAX_MTB_REDUCED` (150), not
local magic numbers.

**Caveat (named, not resolved):** this is a pooled-variance pre-filter, computed across all
primary samples before any group split. A feature whose variance is driven by a real
between-group difference belongs in the top-variance set for a good reason, but a feature
with high *within-Healthy* variance for technical reasons can also make the cut ahead of a
smaller, perfectly real group difference. That is a mild selection bias on every downstream
comparison in this notebook. Polyamines and their proxies are exempt — they are always
retained, so the study's pre-specified targets never depend on this filter.

The reduction is computed on the **primary** samples only, so the sensitivity group cannot
influence which features the whole pipeline keeps.

In [ ]:
for ds in DATASETS_ANALYSIS:
    spc_cols = [c for c in transformed_species[ds].columns if c != "Sample"]
    mtb_cols = [c for c in transformed_mtb[ds].columns     if c != "Sample"]

    # Polyamines AND their pre-specified proxies are protected from both filters.
    pa_cols = (list(polyamine_columns.get(ds, {}).values())
               + [col for col, _via in polyamine_proxies.get(ds, {}).values()])

    # Species: top-variance selection, polyamine-pathway-agnostic (no target leakage)
    top_spc = (transformed_species[ds][spc_cols]
               .var().nlargest(MAX_SPECIES_REDUCED).index.tolist())
    transformed_species[ds] = transformed_species[ds][["Sample"] + top_spc]

    # Metabolites: top-variance selection + polyamine/proxy protection
    top_mtb = (transformed_mtb[ds][mtb_cols]
               .var().nlargest(MAX_MTB_REDUCED).index.tolist())
    top_mtb = list(dict.fromkeys(top_mtb + [c for c in pa_cols if c in mtb_cols]))
    transformed_mtb[ds] = transformed_mtb[ds][["Sample"] + top_mtb]

    # Keep the sensitivity tables on the SAME feature set, so NB06 can project
    # those samples onto factors learned from the primary set.
    if ds in sensitivity_species:
        sensitivity_species[ds] = sensitivity_species[ds][["Sample"] + top_spc]
        sensitivity_mtb[ds]     = sensitivity_mtb[ds][["Sample"] + top_mtb]

    n_kept_pa = sum(1 for c in pa_cols if c in top_mtb)
    print(f"{ds}: species {len(spc_cols)} -> {len(top_spc)} | "
          f"metabolites {len(mtb_cols)} -> {len(top_mtb)} "
          f"({n_kept_pa}/{len(pa_cols)} polyamine+proxy columns protected)")

# Undetected-metabolite mask (pre-imputation value NaN or <= 0) for the retained
# metabolites -- NB06 uses it to give MOFA+ genuine missing values instead of the
# half-min/log/centred floor. Not defined for pre-scaled tables, where "<= 0" has
# no detection meaning (a z-score below zero is simply below average).
undetected_mtb = {}
for ds in DATASETS_ANALYSIS:
    if MTB_PRESCALED.get(ds):
        print(f"{ds}: pre-scaled metabolites -- no undetected mask (<=0 is not 'undetected')")
        continue
    fm = preprocessed['filtered_mtb'][ds].set_index('Sample')
    keep_cols = [c for c in transformed_mtb[ds].columns if c != 'Sample' and c in fm.columns]
    vals = fm.loc[transformed_mtb[ds]['Sample'], keep_cols]
    und = (vals.isna() | (vals <= 0)).reset_index()
    undetected_mtb[ds] = und
    print(f"{ds}: undetected metabolite cells = {und[keep_cols].values.mean():.1%}")

# Save reduced data so NB03+ load the same feature set AND the same sample set.
reduced_pkl_path = RESULTS_DIR / "intermediate" / "preprocessed_data_reduced_v2.pkl"
with open(reduced_pkl_path, "wb") as f:
    pickle.dump({
        # Primary analysis set -- the default keys every downstream notebook reads.
        "transformed_species": transformed_species,
        "transformed_mtb":     transformed_mtb,
        "harmonized_meta":     harmonized_meta,
        # Sensitivity group, same features, separate keys: opt-in only (NB06 projects it).
        "sensitivity_species": sensitivity_species,
        "sensitivity_mtb":     sensitivity_mtb,
        "sensitivity_meta":    sensitivity_meta,
        "polyamine_columns":   polyamine_columns,
        "polyamine_proxies":   polyamine_proxies,
        "mtb_kegg_map":        mtb_kegg_map,
        "datasets":            DATASETS_ANALYSIS,
        "dataset_discovery":   DATASET_DISCOVERY,
        "datasets_validation": DATASETS_VALIDATION,
        "undetected_mtb":      undetected_mtb,
        "mtb_prescaled":       MTB_PRESCALED,
        "stage_group_order":   STAGE_GROUP_ORDER,
        "sample_scope":        "primary (utils.primary_mask); sensitivity group under sensitivity_* keys",
        "feature_reduction":   {"method": "variance_topN",
                                 "MAX_SPECIES_REDUCED": MAX_SPECIES_REDUCED,
                                 "MAX_MTB_REDUCED": MAX_MTB_REDUCED,
                                 "protected": "polyamines + pre-specified proxies"},
    }, f)
print(f"\nReduced pkl saved -> {reduced_pkl_path} "
      f"({reduced_pkl_path.stat().st_size / 1e6:.1f} MB)")
print(f"  primary samples per cohort     : "
      f"{ {ds: len(transformed_species[ds]) for ds in DATASETS_ANALYSIS} }")
print(f"  sensitivity samples held aside : "
      f"{ {ds: len(v) for ds, v in sensitivity_species.items()} }")

---
## Step 6A: Residual / Confound Diagnostics

How much variance in the transformed species data is explained by sample-level covariates?
A high per-feature R² is what motivates adjusting for them, which Step 6B now does
(`utils.clr_lm_da`). Purely diagnostic — it does not modify `transformed_species` or the
pickle.

Covariates: the full Yachida set (`utils.YACHIDA_COMPLETE_CONFOUNDERS` — Age, BMI, Gender,
Brinkman.Index, Alcohol) plus `Stage.Group`, so the diagnostic reports on the same variables
the adjusted DA model uses. Tumour location is *not* among them: it is "-" for every
non-cancer sample, so on this mixed set it would simply re-encode disease status.

**Fix applied:** the old notebook quoted a formula term in `Q("...")` only if the column name
contained a *space*. `Study.Group` contains a `.`, not a space, so patsy tried to parse it as
`Study` attribute-accessed `.Group` — a parse error on *every* feature. A bare `except: pass`
swallowed them all, so the notebook reported a fabricated R²=0.000 for every dataset without
ever fitting a regression. This version builds every formula with `utils.build_formula()`
(which quotes any name containing anything other than letters/digits/underscore) and counts
and prints fit failures instead of swallowing them.

In [ ]:
CONFOUNDERS    = YACHIDA_COMPLETE_CONFOUNDERS + ['Stage.Group']
MAX_FEAT_RESID = 100   # test top-variance species (speed)

for ds in DATASETS_ANALYSIS:
    meta     = harmonized_meta[ds].copy()
    spc      = transformed_species[ds]
    spc_cols = [c for c in spc.columns if c != 'Sample']
    top_spc  = spc[spc_cols].var().nlargest(MAX_FEAT_RESID).index.tolist()
    merged   = meta.merge(spc[['Sample'] + top_spc], on='Sample')

    avail = [c for c in CONFOUNDERS if c in merged.columns
             and merged[c].notna().mean() > 0.3
             and merged[c].nunique(dropna=True) > 1]
    if len(avail) < 1:
        print(f'{ds}: insufficient confounder data -- skip'); continue

    # NOTE: categoricals (Gender, Stage.Group) are left as strings on purpose.
    # patsy auto-detects object-dtype columns and dummy-encodes them inside the
    # formula; the old code pd.factorize()'d them into arbitrary integer codes and
    # fit them as continuous, which is wrong for a nominal variable.

    r2_vals, n_failed, fail_msgs = [], 0, []
    for feat in top_spc:
        sub = merged[avail + [feat]].dropna()
        if len(sub) < 10:
            continue
        formula = build_formula(feat, avail)
        try:
            r2_vals.append(sm_ols(formula, data=sub).fit().rsquared)
        except Exception as e:
            n_failed += 1
            if len(fail_msgs) < 3:
                fail_msgs.append(f'{feat[:40]}: {e}')

    status = 'none -- all fits succeeded' if n_failed == 0 else 'see examples below'
    print(f'{ds}: {n_failed} / {len(top_spc)} OLS fits failed ({status})')
    for msg in fail_msgs:
        print(f'    FAILED: {msg[:120]}')

    r2_arr = np.array(r2_vals) if r2_vals else np.array([0.0])
    print(f'{ds} -- Confounder R2 (top-{len(top_spc)} species, adjusting for {avail}):')
    print(f'  mean={r2_arr.mean():.3f}  median={np.median(r2_arr):.3f}  '
          f'max={r2_arr.max():.3f}  >0.10: {(r2_arr > 0.1).mean():.1%}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(r2_arr, bins=20, color='#3498db', edgecolor='white', alpha=0.85)
    axes[0].axvline(0.1, color='red', lw=1.5, ls='--', label='R2=0.10')
    axes[0].set_xlabel('Confounder R2 per species', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_title(f'{ds} Variance explained by covariates', fontsize=10, fontweight='bold')
    axes[0].legend()

    probplot(spc[top_spc[0]].dropna(), plot=axes[1])
    axes[1].set_title(f'Q-Q: {top_spc[0][:40]} (highest-variance species)', fontsize=10, fontweight='bold')

    plt.suptitle(f'{ds} -- Residual / Confound Diagnostics', fontsize=12, fontweight='bold')
    plt.tight_layout()
    out_r = RESULTS_DIR / 'figures' / f'residual_diagnostics_{ds}_v2.png'
    os.makedirs(out_r.parent, exist_ok=True)
    plt.savefig(out_r, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Saved: {out_r}')

---
## Step 6: Differential Abundance (fine-grained `Study.Group` scheme), with FDR Correction

Uses `utils.COMPARISONS` (per-dataset lists of `(group1, group2)` pairs) and
`utils.differential_abundance()`, which now reports a linear-space **`MeanDiff`**
column (`MeanGroup2 - MeanGroup1`), not the old `Log2FC = log2((mean2+eps)/(mean1+eps))`
-- that extra log-of-a-ratio-of-already-transformed-values was mathematically invalid
whenever a group mean was negative (routine for CLR/log-transformed data) and produced
NaN fold-changes throughout the old notebook's output.

In [ ]:
da_results = {}

for ds in DATASETS_ANALYSIS:
    da_results[ds] = {}
    for group1, group2 in COMPARISONS[ds]:
        print(f'\nRunning DA: {ds} -- {group1} vs {group2}')
        spc_da = differential_abundance(
            transformed_species[ds], harmonized_meta[ds], group1, group2)
        mtb_da = differential_abundance(
            transformed_mtb[ds], harmonized_meta[ds], group1, group2)
        da_results[ds][(group1, group2)] = {'species': spc_da, 'metabolites': mtb_da}
        n_spc = spc_da['Significant'].sum() if not spc_da.empty else 0
        n_mtb = mtb_da['Significant'].sum() if not mtb_da.empty else 0
        print(f'  Species: {n_spc} significant (q<0.05)  |  Metabolites: {n_mtb} significant')

print('\nDifferential abundance complete.')

In [ ]:
# Volcano plots -- metabolites (highlight polyamines); fc_col='MeanDiff' (see Step 6 note)
for ds in DATASETS_ANALYSIS:
    pa_features = list(polyamine_columns.get(ds, {}).values())
    for (g1, g2), results in da_results[ds].items():
        if results['metabolites'].empty:
            continue
        fig, ax = volcano_plot(
            results['metabolites'],
            title=f'{ds} -- Metabolites: {g1} vs {g2}',
            fc_col='MeanDiff',
            highlight_features=pa_features,
            save_path=RESULTS_DIR / 'figures' / 'differential' / f'{ds}_{g1}_vs_{g2}_mtb_volcano_v2.png'
        )
        plt.show()
        plt.close()

In [ ]:
# Volcano plots -- species
for ds in DATASETS_ANALYSIS:
    for (g1, g2), results in da_results[ds].items():
        if results['species'].empty:
            continue
        fig, ax = volcano_plot(
            results['species'],
            title=f'{ds} -- Species: {g1} vs {g2}',
            fc_col='MeanDiff',
            save_path=RESULTS_DIR / 'figures' / 'differential' / f'{ds}_{g1}_vs_{g2}_spc_volcano_v2.png'
        )
        plt.show()
        plt.close()

In [ ]:
# Polyamine-specific DA results table
pa_da_rows = []
for ds in POLYAMINE_DATASETS_ANALYSIS:
    if ds not in da_results:
        continue
    for (g1, g2), results in da_results[ds].items():
        mtb_da = results['metabolites']
        if mtb_da.empty:
            continue
        for kegg_id, col_name in polyamine_columns[ds].items():
            row = mtb_da[mtb_da['Feature'] == col_name]
            if not row.empty:
                pa_da_rows.append({
                    'Dataset':     ds,
                    'Comparison':  f'{g1} vs {g2}',
                    'Polyamine':   POLYAMINE_KEGG[kegg_id],
                    'MeanDiff':    row['MeanDiff'].values[0],
                    'QValue':      row['QValue'].values[0],
                    'Significant': row['Significant'].values[0],
                })

pa_da_df = pd.DataFrame(pa_da_rows)
display(pa_da_df)
pa_da_df.to_csv(RESULTS_DIR / 'tables' / 'polyamine_DA_summary_v2.csv', index=False)

# v2.1: the fine-grained, UNADJUSTED DA family is no longer written to disk.
# It produced 10 per-comparison CSVs under the SAME filename template as the
# confounder-adjusted grouped family (Step 6B), so Yachida carried two
# statistically different answers to one question under near-identical names --
# and since Stage_0 u Stage_I_II == Early_CRC the two overlap by construction.
# A read-vs-write cross-reference found no reader for any of them: NB03 and the
# ML workbench both read the adjusted tables/differential_abundance_v2.csv.
# `da_results` stays in memory for the volcano plots and the summary below.
print('Fine-grained DA computed and kept in memory (not written to disk).')
print('The single DA output is tables/differential_abundance_v2.csv, written by')
print('Step 6B with confounder-adjusted statistics. See CONTRACTS.md.')

---
## Step 6B: Differential Abundance (grouped `Stage.Group` scheme) — **both tests**

`utils.COMPARISONS_GROUP["YACHIDA-CRC-2019"]` defines three comparisons on the corrected
grouping (`utils.STAGE_GROUP_MAP`): `Healthy`, `Early_CRC` (Stage_0 + Stage_I_II),
`Advanced_CRC` (Stage_III_IV). Only Yachida has `Stage.Group` populated (Sinha and Kim have
no staging), so this step is Yachida-only.

Each comparison is tested twice and the results are merged:

1. **Mann-Whitney U + Cliff's δ** (`utils.differential_abundance`) — unadjusted, rank-based.
   `MeanDiff = mean(group2) − mean(group1)`, so **MeanDiff > 0 means higher in group2**;
   `CliffsD` carries the **opposite** sign by construction.
2. **Covariate-adjusted linear model** (`utils.clr_lm_da`) — adjusts for Age, BMI, Gender,
   Brinkman.Index and Alcohol. For **species** it runs on CLR values with the LinDA
   compositional bias correction; for **metabolites** it runs on the log10 matrix with
   `bias_correct=False`, because a metabolite panel is not a composition and subtracting the
   coefficient mode would not be meaningful there. `Coef > 0` means higher in group2, i.e.
   the **same** sign convention as `MeanDiff`.

### Robust hits

`Robust == True` requires **q < 0.05 in both tests AND the same direction**. That is the
column NB03 selects its ML targets from, and the one NB08/NB09 treat as statistical
evidence. Features significant in only one test are kept in the table with `Robust = False`
and an `Agreement` label, so a reader can see what confounding adjustment removed rather
than having it silently dropped:

- `both_q_same_dir` — robust.
- `both_q_opposite_dir` — significant in both but directions disagree: almost always a
  confounded feature, never counted as a hit.
- `mw_only` — passes the rank test, fails adjustment. The prime suspects are the
  smoking-associated taxa, since Brinkman index differs ~2–4× across these groups.
- `lm_only` — adjustment reveals an effect the unadjusted test missed (confounding was
  masking it).

Per-comparison CSVs: `{ds}_{g1}_vs_{g2}_{species|metabolites}_DA_v2.csv` keeps the
Mann-Whitney columns NB03 and the plotting code already expect, with the model columns and
`Robust` joined on.

In [ ]:
GROUP_DS = DATASET_DISCOVERY   # the only cohort in utils.COMPARISONS_GROUP

# Confounders actually usable in this cohort (present, non-constant, mostly observed).
_meta_g = harmonized_meta[GROUP_DS]
DA_COVARS = [c for c in YACHIDA_COMPLETE_CONFOUNDERS
             if c in _meta_g.columns
             and _meta_g[c].notna().mean() > 0.5
             and _meta_g[c].nunique(dropna=True) > 1]
print(f'Adjusted model covariates: {DA_COVARS}')

da_group_da_spc = {}
da_group_da_mtb = {}
robust_summary  = []


def merge_da(mw, lm, label, dtype):
    """Join the rank test and the adjusted model on Feature and classify agreement.

    Sign conventions are aligned first: MeanDiff and Coef are both
    'group2 - group1', so a robust hit needs sign(MeanDiff) == sign(Coef).
    """
    if mw is None or mw.empty:
        return pd.DataFrame()
    out = mw.copy()
    if lm is None or lm.empty:
        out['Coef_adj'] = np.nan; out['SE_adj'] = np.nan
        out['PValue_adj'] = np.nan; out['QValue_adj'] = np.nan
        out['Agreement'] = 'mw_only_no_model'
        out['Robust'] = False
        return out

    l = lm[['Feature', 'Coef', 'SE', 'PValue', 'QValue']].rename(columns={
        'Coef': 'Coef_adj', 'SE': 'SE_adj',
        'PValue': 'PValue_adj', 'QValue': 'QValue_adj'})
    out = out.merge(l, on='Feature', how='left')

    sig_mw = out['QValue'] < FDR_THRESHOLD
    sig_lm = out['QValue_adj'] < FDR_THRESHOLD
    same   = np.sign(out['MeanDiff']) == np.sign(out['Coef_adj'])

    out['Agreement'] = np.select(
        [sig_mw & sig_lm & same, sig_mw & sig_lm & ~same, sig_mw & ~sig_lm, ~sig_mw & sig_lm],
        ['both_q_same_dir', 'both_q_opposite_dir', 'mw_only', 'lm_only'],
        default='neither')
    out['Robust'] = out['Agreement'] == 'both_q_same_dir'
    return out


for group1, group2 in COMPARISONS_GROUP[GROUP_DS]:
    print(f'\n=== {GROUP_DS}: {group1} vs {group2} ===')

    # 1. Unadjusted rank test
    mw_spc = differential_abundance(transformed_species[GROUP_DS], harmonized_meta[GROUP_DS],
                                    group1, group2, group_col='Stage.Group')
    mw_mtb = differential_abundance(transformed_mtb[GROUP_DS], harmonized_meta[GROUP_DS],
                                    group1, group2, group_col='Stage.Group')

    # 2. Adjusted model. Species: compositional (CLR in, LinDA bias correction).
    #    Metabolites: log10 matrix, NOT a composition -> bias_correct=False.
    lm_spc = clr_lm_da(transformed_species[GROUP_DS], harmonized_meta[GROUP_DS],
                       group1, group2, group_col='Stage.Group',
                       covars=DA_COVARS, already_clr=True, bias_correct=True)
    lm_mtb = clr_lm_da(transformed_mtb[GROUP_DS], harmonized_meta[GROUP_DS],
                       group1, group2, group_col='Stage.Group',
                       covars=DA_COVARS, already_clr=True, bias_correct=False)

    spc_da = merge_da(mw_spc, lm_spc, f'{group1}_vs_{group2}', 'species')
    mtb_da = merge_da(mw_mtb, lm_mtb, f'{group1}_vs_{group2}', 'metabolites')

    da_group_da_spc[(group1, group2)] = spc_da
    da_group_da_mtb[(group1, group2)] = mtb_da

    for dtype, df_da, lm_res in (('species', spc_da, lm_spc), ('metabolites', mtb_da, lm_mtb)):
        if df_da.empty:
            print(f'  {dtype}: no testable features')
            continue
        counts = df_da['Agreement'].value_counts().to_dict()
        n_rob = int(df_da['Robust'].sum())
        n_mw  = int((df_da['QValue'] < FDR_THRESHOLD).sum())
        n_lm  = int((df_da['QValue_adj'] < FDR_THRESHOLD).sum())
        print(f'  {dtype}: MW q<0.05: {n_mw} | adjusted q<0.05: {n_lm} | '
              f'ROBUST (both + same dir): {n_rob}')
        print(f'    agreement breakdown: {counts}')
        if not lm_res.empty:
            print(f"    model: covars={lm_res.attrs.get('covariates')}, "
                  f"dropped {lm_res.attrs.get('n_dropped_missing_covar', 0)} row(s) for missing covars, "
                  f"bias offset={lm_res.attrs.get('bias_offset', 0.0):+.4f}")

        fname = f'{GROUP_DS}_{group1}_vs_{group2}_{dtype}_DA_v2.csv'
        df_da.to_csv(RESULTS_DIR / 'tables' / fname, index=False)

        robust_summary.append({
            'Comparison': f'{group1}_vs_{group2}', 'Type': dtype,
            'N_tested': len(df_da), 'N_MW_q05': n_mw, 'N_adjusted_q05': n_lm,
            'N_robust': n_rob,
            'N_mw_only': int((df_da['Agreement'] == 'mw_only').sum()),
            'N_lm_only': int((df_da['Agreement'] == 'lm_only').sum()),
            'N_opposite': int((df_da['Agreement'] == 'both_q_opposite_dir').sum()),
        })

robust_df = pd.DataFrame(robust_summary)
robust_df.to_csv(RESULTS_DIR / 'tables' / 'da_robust_hit_summary_v2.csv', index=False)
print('\n=== Robust-hit summary (both tests, same direction) ===')
print(robust_df.to_string(index=False))
print('\nSaved: da_robust_hit_summary_v2.csv')
print('\nInterpretation: "mw_only" counts features whose unadjusted significance does not '
      'survive adjustment for age/BMI/sex/smoking/alcohol. In this cohort smoking is the '
      'main suspect (median Brinkman 100 in Healthy vs 196-390 in CRC). Those features are '
      'NOT treated as findings anywhere downstream.')

In [ ]:
# Volcano plots for the grouped scheme (metabolites, polyamines highlighted)
pa_features_group = list(polyamine_columns.get(GROUP_DS, {}).values())
for (g1, g2), mtb_da in da_group_da_mtb.items():
    if mtb_da.empty:
        continue
    fig, ax = volcano_plot(
        mtb_da,
        title=f'{GROUP_DS} (grouped) -- Metabolites: {g1} vs {g2}',
        fc_col='MeanDiff',
        highlight_features=pa_features_group,
        save_path=RESULTS_DIR / 'figures' / 'differential' / f'{GROUP_DS}_{g1}_vs_{g2}_mtb_volcano_v2.png'
    )
    plt.show()
    plt.close()

for (g1, g2), spc_da in da_group_da_spc.items():
    if spc_da.empty:
        continue
    fig, ax = volcano_plot(
        spc_da,
        title=f'{GROUP_DS} (grouped) -- Species: {g1} vs {g2}',
        fc_col='MeanDiff',
        save_path=RESULTS_DIR / 'figures' / 'differential' / f'{GROUP_DS}_{g1}_vs_{g2}_spc_volcano_v2.png'
    )
    plt.show()
    plt.close()

---
## Step 7: Correlation Analysis

### 7.1 Species-Polyamine Correlations (per dataset)

In [ ]:
polyamine_corr = {}

for ds in POLYAMINE_DATASETS_ANALYSIS:
    if ds not in DATASETS_ANALYSIS:
        continue
    target_mtbs = list(polyamine_columns[ds].values())
    print(f'\nComputing correlations for {ds} ({len(target_mtbs)} polyamines)...')

    corr_df = compute_correlations(
        transformed_species[ds], transformed_mtb[ds],
        target_metabolites=target_mtbs
    )
    polyamine_corr[ds] = corr_df

    if not corr_df.empty:
        n_sig = (corr_df['QValue'] < 0.05).sum()
        print(f'  {len(corr_df)} pairs tested, {n_sig} significant (q<0.05)')
        sig_only = corr_df[corr_df['QValue'] < 0.05]
        top = sig_only.reindex(sig_only['Rho'].abs().sort_values(ascending=False).index).head(5)
        for _, r in top.iterrows():
            print(f"    {r['Species'][:40]} <-> {r['Metabolite']}: "
                  f"rho={r['Rho']:.3f}, q={r['QValue']:.2e}")

In [ ]:
# Heatmaps: top species correlated with polyamines
for ds in POLYAMINE_DATASETS_ANALYSIS:
    if not polyamine_corr.get(ds, pd.DataFrame()).empty:
        fig, ax = plot_correlation_heatmap(
            polyamine_corr[ds],
            title=f'{ds} -- Species vs Polyamines (top 20, q<0.05)',
            top_n=20,
            save_path=RESULTS_DIR / 'figures' / 'correlation' / f'{ds}_polyamine_heatmap_v2.png'
        )
        if fig:
            plt.show()
            plt.close()

### 7.2 Full Species-Metabolite Correlations

Species subset is determined by top variance **plus any species already significant in
7.1**, to avoid excluding consistent low-variance producers.

In [ ]:
full_corr = {}
MAX_SPC_FEATURES = 200   # correlation-compute-cost cap (separate from the global
MAX_MTB_FEATURES = 500   # top-variance reduction applied earlier in the notebook)

for ds in DATASETS_ANALYSIS:
    spc_cols = [c for c in transformed_species[ds].columns if c != 'Sample']
    mtb_cols = [c for c in transformed_mtb[ds].columns if c != 'Sample']

    # Top-variance species, but always keep species significant in 7.1
    if len(spc_cols) > MAX_SPC_FEATURES:
        spc_top_var = (transformed_species[ds][spc_cols].var()
                       .nlargest(MAX_SPC_FEATURES).index.tolist())
        if ds in polyamine_corr and not polyamine_corr[ds].empty:
            sig_spc = polyamine_corr[ds][polyamine_corr[ds]['QValue'] < 0.05]['Species'].unique()
            for s in sig_spc:
                if s not in spc_top_var:
                    spc_top_var.append(s)
        spc_df = transformed_species[ds][['Sample'] + spc_top_var]
    else:
        spc_df = transformed_species[ds]

    # Top-variance metabolites, always protect polyamines
    if len(mtb_cols) > MAX_MTB_FEATURES:
        mtb_top_var = (transformed_mtb[ds][mtb_cols].var()
                       .nlargest(MAX_MTB_FEATURES).index.tolist())
        for pa_col in polyamine_columns.get(ds, {}).values():
            if pa_col not in mtb_top_var:
                mtb_top_var.append(pa_col)
        mtb_df = transformed_mtb[ds][['Sample'] + mtb_top_var]
    else:
        mtb_df = transformed_mtb[ds]

    print(f'\n{ds}: computing correlations '
          f'({len(spc_df.columns)-1} species x {len(mtb_df.columns)-1} metabolites)...')
    corr_df = compute_correlations(spc_df, mtb_df)
    full_corr[ds] = corr_df

    if not corr_df.empty:
        n_sig = (corr_df['QValue'] < 0.05).sum()
        print(f'  {len(corr_df)} pairs tested, {n_sig} significant (q<0.05)')

### 7.3 Partial Correlations (controlling for the full confounder set)

Adjusts species–polyamine Spearman correlations for the complete Yachida confounder set:
`Age`, `BMI`, `Gender`, `Brinkman.Index` and `Alcohol` (`utils.YACHIDA_COMPLETE_CONFOUNDERS`),
all carried through `harmonized_meta` by `utils.harmonize_metadata()`.

**Tumour location is deliberately not a covariate here.** `Tumor location` is "-" for every
non-cancer sample, so on this mixed Healthy+CRC set adjusting for it would partial out
disease status itself. `utils.build_confounder_matrix()` refuses the request outright rather
than relying on this note; the within-cancer analysis that *may* legitimately adjust for
tumour site is NB04.

**Missing covariates are dropped, not filled.** The previous version replaced missing
`Brinkman.Index` and `Alcohol` with the column mean before adjusting. Mean-filling a
covariate and then partialling it out understates the uncertainty and biases the adjustment
toward no effect for exactly the samples where the covariate was unknown. Complete-case
analysis is used instead, and the number of dropped samples is reported.

All reduced species are tested against every polyamine — no marginal pre-screen, which would
make the partial-correlation FDR conditional on a prior screen. The table is saved as
`{dataset}_polyamine_partial_correlations_v2.csv`; **NB09 uses it as its SHAP-independent
candidate pool**, which is what breaks the old pipeline's circularity (SHAP picking the
candidates and then being counted as evidence for them).

In [ ]:
try:
    import pingouin as pg
    HAS_PINGOUIN = True
    print(f'pingouin {pg.__version__} loaded.')
except ImportError:
    print('pingouin not installed. Run: pip install pingouin')
    HAS_PINGOUIN = False

partial_corr_results = {}

if HAS_PINGOUIN:
    for ds in POLYAMINE_DATASETS_ANALYSIS:
        if ds not in DATASETS_ANALYSIS:
            continue
        print(f'\n--- Partial correlations: {ds} ---')
        meta = harmonized_meta[ds].copy()

        # Build the covariate list from what this cohort actually has. Gender is
        # encoded numerically; everything else must already be numeric.
        covariates = []
        for col in YACHIDA_COMPLETE_CONFOUNDERS:
            if col not in meta.columns:
                continue
            if col == 'Gender':
                meta['Gender_num'] = (meta['Gender'].astype(str).str.lower() == 'male').astype(float)
                if meta['Gender_num'].notna().mean() > 0.5 and meta['Gender_num'].nunique() > 1:
                    covariates.append('Gender_num')
                continue
            series = pd.to_numeric(meta[col], errors='coerce')
            if series.notna().mean() > 0.5 and series.nunique(dropna=True) > 1:
                meta[col] = series
                covariates.append(col)

        # No tumour-location covariate: it is a disease-status proxy on this set.

        if not covariates:
            print('  No usable covariates -- skipping.')
            continue
        print(f'  Adjusting for: {covariates}')

        spc_renamed = transformed_species[ds].rename(
            columns={c: f'SPC__{c}' for c in transformed_species[ds].columns if c != 'Sample'})
        mtb_renamed = transformed_mtb[ds].rename(
            columns={c: f'MTB__{c}' for c in transformed_mtb[ds].columns if c != 'Sample'})

        merged_all = spc_renamed.merge(mtb_renamed, on='Sample').merge(
            meta[['Sample'] + covariates], on='Sample')
        # Complete-case: drop rows missing any covariate (never mean-fill).
        merged = merged_all.dropna(subset=covariates)
        n_dropped = len(merged_all) - len(merged)
        print(f'  n={len(merged)} complete cases'
              + (f' ({n_dropped} dropped for missing covariates)' if n_dropped else ''))

        top_species = [c for c in transformed_species[ds].columns if c != 'Sample']

        # Direct polyamines plus pre-specified proxies; each row records which it was.
        targets = [(col, POLYAMINE_KEGG[k], 'direct')
                   for k, col in polyamine_columns[ds].items()]
        targets += [(col, POLYAMINE_KEGG[k], f'proxy:{via}')
                    for k, (col, via) in polyamine_proxies.get(ds, {}).items()]

        pc_rows = []
        for spc in top_species:
            spc_key = f'SPC__{spc}'
            if spc_key not in merged.columns:
                continue
            for pa_col, pa_name, measured_as in targets:
                pa_key = f'MTB__{pa_col}'
                if pa_key not in merged.columns:
                    continue
                try:
                    result = pg.partial_corr(data=merged, x=spc_key, y=pa_key,
                                              covar=covariates, method='spearman')
                except Exception:
                    continue
                pc_rows.append({
                    'Species':        spc,
                    'Metabolite':     pa_col,
                    'Polyamine':      pa_name,
                    'Measured_as':    measured_as,
                    'Rho_partial':    result['r'].values[0],
                    'PValue_partial': result['p-val'].values[0],
                    'N':              int(result['n'].values[0]),
                })

        if not pc_rows:
            print('  No valid pairs computed.')
            continue

        pc_df = pd.DataFrame(pc_rows)
        _, qvals, _, _ = multipletests(pc_df['PValue_partial'], method='fdr_bh')
        pc_df['QValue_partial'] = qvals
        col_to_kegg = {v: k for k, v in polyamine_columns[ds].items()}
        col_to_kegg.update({col: k for k, (col, _v) in polyamine_proxies.get(ds, {}).items()})
        pc_df['KEGG'] = pc_df['Metabolite'].map(col_to_kegg)
        pc_df['Covariates'] = ', '.join(covariates)
        partial_corr_results[ds] = pc_df
        pc_df.to_csv(RESULTS_DIR / 'tables' / f'{ds}_polyamine_partial_correlations_v2.csv',
                     index=False)

        n_sig = int((pc_df['QValue_partial'] < 0.05).sum())
        n_pos = int(((pc_df['QValue_partial'] < 0.05) & (pc_df['Rho_partial'] > 0)).sum())
        print(f'  {len(pc_df)} pairs tested, {n_sig} significant (q<0.05), '
              f'of which {n_pos} POSITIVE (candidate producers for NB09)')

### 7.4 Cross-Cohort Replication (Yachida vs Sinha vs Kim, genus level)

**Why genus, not species.** Yachida is shotgun metagenomics at species resolution; Sinha and
Kim are 16S at genus resolution. Species names therefore *cannot* match across cohorts — the
earlier species-level version of this step returned zero pairs for that reason alone, which
looks like a negative result but is an artefact of the taxonomy. All three cohorts are
aggregated to **genus** (GTDB placeholder suffixes such as `_A` stripped, then re-aggregated),
CLR-transformed, and correlated with each polyamine.

**Why KEGG IDs, not column names.** The three platforms name the same compound differently
(`C00134_Putrescine` / `PUTRESCINE` / `N-acetylputrescine`), so matching is done on KEGG
compound ID.

**Proxy handling.** Kim does not measure putrescine or cadaverine, only their N-acetyl
derivatives. Those are mapped to the **parent** compound for matching purposes — otherwise
Kim could never replicate the study's headline compound — but every resulting row carries a
`Measured_as` / `Proxy_used` label. A replication resting on a derivative is weaker evidence
than one resting on the parent, and the final evidence table in NB09 keeps that distinction
rather than averaging it away.

**Replication is directional, not just "both significant".** `Same_direction` requires the
sign of ρ to agree across the cohorts in which the pair is significant. A pair significant in
two cohorts with *opposite* signs is not a replication; it is a contradiction, and is
reported as such.

Sinha and Kim metabolites are pre-standardised; Spearman is rank-based, so that has no
effect on this step.

In [ ]:
import re
from utils import aggregate_to_genus, clr_transform, filter_by_prevalence, PREVALENCE_SPECIES

def _norm_genus(g):
    g = re.sub(r'_[A-Z]+$', '', str(g).strip())      # GTDB placeholder suffixes
    return g.strip('[]').strip()

def genus_clr(ds):
    """Genus-level CLR table for one cohort, restricted to the primary samples."""
    src = preprocessed['filtered_species'][ds]
    keep = set(harmonized_meta[ds]['Sample'])
    src = src[src['Sample'].isin(keep)].reset_index(drop=True)

    gen = aggregate_to_genus(src)
    feat = gen.drop(columns='Sample')
    feat.columns = [_norm_genus(c) for c in feat.columns]
    feat = feat.T.groupby(level=0).sum().T          # merge genera joined by suffix stripping
    feat = feat.loc[:, ~feat.columns.isin(['', 'Unknown', 'None'])]
    feat.insert(0, 'Sample', gen['Sample'].values)
    feat = filter_by_prevalence(feat, threshold=PREVALENCE_SPECIES)
    return clr_transform(feat)

# Per-cohort map {metabolite column -> PARENT polyamine KEGG id}, so a proxy column
# matches its parent across cohorts. Also record which cohorts used a proxy.
pa_match_map = {}
proxy_used_by_ds = {}
for ds in POLYAMINE_DATASETS_ANALYSIS:
    m = {col: kegg for kegg, col in polyamine_columns.get(ds, {}).items()}
    px = {}
    for kegg, (col, via) in polyamine_proxies.get(ds, {}).items():
        m[col] = kegg          # map derivative -> PARENT compound
        px[kegg] = via
    pa_match_map[ds] = m
    proxy_used_by_ds[ds] = px

genus_polyamine_corr = {}
for ds in POLYAMINE_DATASETS_ANALYSIS:
    pa_cols = list(pa_match_map[ds].keys())
    if not pa_cols:
        print(f'{ds}: no polyamine columns -- skipped')
        continue
    gclr = genus_clr(ds)
    corr = compute_correlations(gclr, transformed_mtb[ds], target_metabolites=pa_cols)
    genus_polyamine_corr[ds] = corr
    n_sig = int((corr['QValue'] < 0.05).sum()) if not corr.empty else 0
    print(f'{ds}: {gclr.shape[1]-1} genera x {len(pa_cols)} polyamine columns, '
          f'{n_sig} pairs q<0.05'
          + (f'  [proxies: {list(proxy_used_by_ds[ds])}]' if proxy_used_by_ds[ds] else ''))

cohorts_present = [ds for ds in genus_polyamine_corr if not genus_polyamine_corr[ds].empty]
if len(cohorts_present) >= 2:
    shared_genera = set.intersection(
        *[set(genus_polyamine_corr[ds]['Species']) for ds in cohorts_present])
    print(f'\nGenera testable in all {len(cohorts_present)} cohorts: {len(shared_genera)}')
else:
    shared_genera = set()
    print('\nFewer than two cohorts have testable pairs -- no replication possible.')


def _rho_lookup(ds):
    km = pa_match_map.get(ds, {})
    return {(r.Species, km.get(r.Metabolite, r.Metabolite)): r.Rho
            for r in genus_polyamine_corr[ds].itertuples()}

def _annotate(df):
    """Add Same_direction, which cohorts replicated, and whether any used a proxy."""
    if df.empty:
        return df
    maps = {ds: _rho_lookup(ds) for ds in cohorts_present}
    df = df.copy()
    same_dir, prox, coh = [], [], []
    for g, k in zip(df['Species'], df['Metabolite_key']):
        signs = {ds: np.sign(m[(g, k)]) for ds, m in maps.items() if (g, k) in m}
        same_dir.append(len(set(signs.values())) == 1 and len(signs) >= 2)
        coh.append(', '.join(sorted(signs)))
        prox.append(', '.join(sorted(
            ds for ds in signs if k in proxy_used_by_ds.get(ds, {}))) or 'none')
    df['Same_direction'] = same_dir
    df['Cohorts_tested'] = coh
    df['Proxy_used_in']  = prox
    df['Polyamine']      = [POLYAMINE_KEGG.get(k, k) for k in df['Metabolite_key']]
    return df

consistent_pairs   = _annotate(find_consistent_correlations(
    genus_polyamine_corr, kegg_map=pa_match_map, min_datasets=2, q_threshold=0.05))
consistent_relaxed = pd.DataFrame()

out_csv = RESULTS_DIR / 'tables' / 'consistent_polyamine_correlations_genus_v2.csv'
if not consistent_pairs.empty:
    n_same = int(consistent_pairs['Same_direction'].sum())
    print(f'\n{len(consistent_pairs)} genus-polyamine pairs (KEGG-matched) significant in '
          f'>=2 cohorts at q<0.05; same direction in all of them: {n_same}')
    if n_same < len(consistent_pairs):
        print(f'  {len(consistent_pairs) - n_same} pair(s) are significant in two cohorts '
              f'with OPPOSITE signs -- contradictions, not replications.')
    display(consistent_pairs.head(20))
    consistent_pairs.rename(columns={'Species': 'Genus'}).to_csv(out_csv, index=False)
    print(f'Saved: {out_csv.name}')
else:
    print('\nNo pairs replicate at q<0.05 in >=2 cohorts. Reporting a q<0.10 '
          'hypothesis-generating list instead, clearly labelled as such.')
    consistent_relaxed = _annotate(find_consistent_correlations(
        genus_polyamine_corr, kegg_map=pa_match_map, min_datasets=2, q_threshold=0.10))
    print(f'At q<0.10: {len(consistent_relaxed)} pairs')
    if not consistent_relaxed.empty:
        consistent_relaxed['Threshold'] = 'q<0.10 (exploratory)'
        display(consistent_relaxed.head(20))
        consistent_relaxed.rename(columns={'Species': 'Genus'}).to_csv(out_csv, index=False)
        print(f'Saved: {out_csv.name}  (exploratory threshold -- NOT a replication claim)')

# Per-cohort genus-level tables, so NB08/NB09 can cite a specific cohort's result.
for ds, corr in genus_polyamine_corr.items():
    if corr.empty:
        continue
    c = corr.copy()
    c['Parent_KEGG'] = c['Metabolite'].map(pa_match_map[ds])
    c['Polyamine']   = c['Parent_KEGG'].map(POLYAMINE_KEGG)
    c['Measured_as'] = np.where(
        c['Parent_KEGG'].isin(proxy_used_by_ds.get(ds, {})), 'proxy', 'direct')
    c.rename(columns={'Species': 'Genus'}).to_csv(
        RESULTS_DIR / 'tables' / f'{ds}_genus_polyamine_correlations_v2.csv', index=False)
print('\nSaved per-cohort genus-level correlation tables.')

In [ ]:
# Cross-cohort dot plot. consistent_pairs carries a KEGG-matched 'Metabolite_key'
# (a PARENT compound id), so for each cohort we look up which of ITS OWN columns map
# to that parent via pa_match_map -- which is also how a proxy column is found.

def _lookup_corr(ds, species, metabolite_key):
    corr_df = genus_polyamine_corr.get(ds)
    if corr_df is None or corr_df.empty:
        return None
    ds_map = pa_match_map.get(ds, {})
    candidate_cols = [col for col, kid in ds_map.items() if kid == metabolite_key]
    if not candidate_cols:
        return None
    match = corr_df[(corr_df['Species'] == species) &
                    (corr_df['Metabolite'].isin(candidate_cols))]
    return None if match.empty else match.iloc[0]


plot_consistent = consistent_pairs if not consistent_pairs.empty else consistent_relaxed

if not plot_consistent.empty:
    top_pairs = plot_consistent.head(15)
    plot_data = []
    for _, pair in top_pairs.iterrows():
        for ds in POLYAMINE_DATASETS_ANALYSIS:
            if ds not in genus_polyamine_corr:
                continue
            row = _lookup_corr(ds, pair['Species'], pair['Metabolite_key'])
            if row is None:
                continue
            is_proxy = pair['Metabolite_key'] in proxy_used_by_ds.get(ds, {})
            plot_data.append({
                'Pair':        f"{pair['Species'][:24]}\n<-> {pair.get('Polyamine', pair['Metabolite_key'])}",
                'Cohort':      ds.split('-')[0].split('_')[0],
                'Rho':         row['Rho'],
                'Significant': row['QValue'] < 0.05,
                'Proxy':       is_proxy,
            })

    if plot_data:
        plot_df = pd.DataFrame(plot_data)
        cohort_colors = {'YACHIDA': '#2a78d6', 'SINHA': '#eb6834', 'Kim': '#2e9e5b'}
        fig, ax = plt.subplots(figsize=(13, max(6, len(top_pairs) * 0.62)))

        for cohort, sub_c in plot_df.groupby('Cohort'):
            col = cohort_colors.get(cohort, '#777777')
            for sig, sub in sub_c.groupby('Significant'):
                # Proxy measurements get a hollow marker: visibly weaker evidence.
                for proxy, s in sub.groupby('Proxy'):
                    ax.scatter(
                        s['Rho'], s['Pair'],
                        facecolors='none' if proxy else col,
                        edgecolors=col, linewidths=1.6,
                        marker='o' if sig else 'x',
                        s=110 if sig else 55, alpha=0.9,
                        label=f"{cohort}{' (proxy)' if proxy else ''}"
                              f"{'' if sig else ' ns'}")

        ax.axvline(0, color='grey', ls='--', alpha=0.5)
        ax.set_xlabel('Spearman rho (genus vs polyamine, CLR)')
        handles, labels = ax.get_legend_handles_labels()
        seen = dict(zip(labels, handles))
        ax.legend(seen.values(), seen.keys(), fontsize=7, loc='best', framealpha=0.9)
        ax.set_title('Cross-cohort replication: genus-polyamine correlations\n'
                     'genus level, matched by KEGG parent compound | filled=q<0.05, '
                     'hollow=measured via N-acetyl proxy, x=not significant',
                     fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'figures' / 'correlation' / 'cross_cohort_consistency_v2.png',
                    dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
    else:
        print('No plottable cross-cohort pairs found.')
else:
    print('No consistent pairs (strict or exploratory) to plot.')

---
## Step 8: Network Analysis

In [ ]:
import networkx as nx

try:
    from community import best_partition
    HAS_COMMUNITY = True
    print('python-louvain loaded.')
except ImportError:
    print('python-louvain not installed. Install with: pip install python-louvain')
    HAS_COMMUNITY = False

os.makedirs(RESULTS_DIR / 'figures' / 'network', exist_ok=True)

In [ ]:
def build_correlation_network(corr_df, rho_threshold=0.3, q_threshold=0.05):
    """Build a bipartite network from significant correlations."""
    G   = nx.Graph()
    sig = corr_df[
        (corr_df['QValue'] < q_threshold) &
        (corr_df['Rho'].abs() >= rho_threshold)
    ]
    for _, row in sig.iterrows():
        G.add_node(row['Species'],    bipartite=0, node_type='species')
        G.add_node(row['Metabolite'], bipartite=1, node_type='metabolite')
        G.add_edge(
            row['Species'], row['Metabolite'],
            weight=abs(row['Rho']),
            sign='positive' if row['Rho'] > 0 else 'negative',
            rho=row['Rho'], qvalue=row['QValue'],
        )
    return G

def compute_network_metrics(G):
    """Compute centrality metrics for network nodes."""
    if len(G) == 0:
        return pd.DataFrame()
    degree      = dict(G.degree())
    betweenness = nx.betweenness_centrality(G)
    return pd.DataFrame({
        'Node':        list(G.nodes()),
        'Degree':      [degree[n]      for n in G.nodes()],
        'Betweenness': [betweenness[n] for n in G.nodes()],
        'NodeType':    [G.nodes[n].get('node_type', 'unknown') for n in G.nodes()],
    }).sort_values('Degree', ascending=False)

print('Network functions defined.')

In [ ]:
# Build networks using full_corr (which includes all polyamines as protected features)
networks        = {}
network_metrics = {}

for ds in DATASETS_ANALYSIS:
    if full_corr.get(ds) is None or full_corr[ds].empty:
        continue
    G = build_correlation_network(full_corr[ds], rho_threshold=0.3, q_threshold=0.05)
    networks[ds]        = G
    network_metrics[ds] = compute_network_metrics(G)

    n_spc = sum(1 for _, d in G.nodes(data=True) if d.get('node_type') == 'species')
    n_mtb = sum(1 for _, d in G.nodes(data=True) if d.get('node_type') == 'metabolite')
    print(f'{ds}: {G.number_of_nodes()} nodes ({n_spc} species, {n_mtb} metabolites), '
          f'{G.number_of_edges()} edges')

In [ ]:
# Community detection (Louvain)
if HAS_COMMUNITY:
    for ds, G in networks.items():
        if len(G) < 3:
            continue
        partition = best_partition(G)
        nx.set_node_attributes(G, partition, 'community')
        communities = [{n for n, c in partition.items() if c == comm}
                        for comm in set(partition.values())]
        try:
            modularity = nx.community.modularity(G, communities)
            print(f'{ds}: {len(communities)} communities, modularity={modularity:.3f}')
        except Exception as e:
            print(f'{ds}: community detection OK but modularity failed: {e}')

In [ ]:
# Polyamine-centric subnetworks
polyamine_networks = {}

for ds in POLYAMINE_DATASETS_ANALYSIS:
    if ds not in networks or len(networks[ds]) == 0:
        continue
    G        = networks[ds]
    pa_nodes = [col for col in polyamine_columns[ds].values() if col in G]

    if not pa_nodes:
        print(f'{ds}: No polyamine nodes in network '
              f'(check that polyamines were included in full_corr computation)')
        continue

    neighbors = set(pa_nodes)
    for n in pa_nodes:
        neighbors.update(G.neighbors(n))
    subG = G.subgraph(neighbors).copy()
    polyamine_networks[ds] = subG
    print(f'{ds}: polyamine subnetwork -- {subG.number_of_nodes()} nodes, '
          f'{subG.number_of_edges()} edges')

In [ ]:
# Visualise polyamine subnetworks
for ds, subG in polyamine_networks.items():
    if subG.number_of_nodes() < 2:
        continue
    pa_node_set = set(polyamine_columns[ds].values())
    fig, ax     = plt.subplots(figsize=(16, 12))
    pos         = nx.spring_layout(subG, k=2, seed=42)

    colors = []
    sizes  = []
    for n in subG.nodes():
        if n in pa_node_set:
            colors.append('red'); sizes.append(400)
        elif subG.nodes[n].get('node_type') == 'species':
            colors.append('skyblue'); sizes.append(150)
        else:
            colors.append('lightgreen'); sizes.append(150)

    edge_colors = ['red' if subG[u][v].get('sign') == 'negative' else 'green'
                   for u, v in subG.edges()]
    nx.draw(subG, pos, node_color=colors, node_size=sizes,
            edge_color=edge_colors, alpha=0.7, ax=ax, with_labels=False)

    labels = {}
    for n in subG.nodes():
        if n in pa_node_set:
            labels[n] = n.split('_', 1)[-1] if '_' in n else n
        elif subG.nodes[n].get('node_type') == 'species':
            labels[n] = n[:25]
    nx.draw_networkx_labels(subG, pos, labels, font_size=7, ax=ax)
    ax.set_title(f'{ds} -- Polyamine-Centric Network\n'
                 f'(red=polyamines, blue=species, green=other metabolites; '
                 f'red edge=negative, green=positive)', fontsize=11)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures' / 'network' / f'{ds}_polyamine_network_v2.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Hub species connected to polyamines (whole-dataset, not stage-stratified)
hub_species_all = []
for ds, subG in polyamine_networks.items():
    pa_node_set   = set(polyamine_columns[ds].values())
    species_nodes = [n for n in subG.nodes() if subG.nodes[n].get('node_type') == 'species']
    for spc in species_nodes:
        pa_neighbors = [nb for nb in subG.neighbors(spc) if nb in pa_node_set]
        if pa_neighbors:
            hub_species_all.append({
                'Dataset':                 ds,
                'Species':                 spc,
                'N_polyamine_connections': len(pa_neighbors),
                'Degree':                  subG.degree(spc),
                'Connected_polyamines':    ', '.join(pa_neighbors),
            })

hub_df = pd.DataFrame(hub_species_all).sort_values('N_polyamine_connections', ascending=False)
display(hub_df.head(20))
hub_df.to_csv(RESULTS_DIR / 'tables' / 'polyamine_hub_species_v2.csv', index=False)

---
## Stratified Hub Species Network Analysis

Separate polyamine-centric correlation networks per CRC stage group
(`YACHIDA-CRC-2019`, the only cohort with staging), using `utils.STAGE_GROUP_ORDER`:
**Healthy, Early_CRC, Advanced_CRC**. MP is excluded from the study and PostSurgery is not
on the severity axis, so neither appears here.

**Caveat — this comparison is confounded by sample size and must not be read as biology.**
The groups differ substantially in n (127 / 96 / 54) while every network uses the same
q<0.05 and |rho|>=0.3 thresholds. Statistical power scales with n, so the Healthy network
will contain more edges than the Advanced_CRC network *by construction*, whether or not its
true correlation structure is any denser. Edge counts across groups are therefore reported
for completeness but are not interpreted as "the network densifies/sparsifies with stage";
no downstream notebook consumes these counts as evidence.

In [ ]:
CRC_DS_HUB = DATASET_DISCOVERY

# Stage.Group comes from utils.harmonize_metadata() (NB01) via utils.STAGE_GROUP_MAP --
# nothing is re-derived locally.
print('Primary stage groups (utils.STAGE_GROUP_ORDER):', STAGE_GROUP_ORDER)
print('utils.STAGE_GROUP_MAP (fine-grained Study.Group -> grouped Stage.Group):')
for sg, grp in STAGE_GROUP_MAP.items():
    note = '   [sensitivity: not analysed here]' if grp not in STAGE_GROUP_ORDER else ''
    print(f'  {sg:14s} -> {grp}{note}')


def filter_by_group(ds, group_name):
    """Species and metabolite frames for samples whose Stage.Group == group_name."""
    meta    = harmonized_meta[ds]
    samples = meta[meta['Stage.Group'] == group_name]['Sample'].values
    spc_sub = transformed_species[ds][
        transformed_species[ds]['Sample'].isin(samples)].reset_index(drop=True)
    mtb_sub = transformed_mtb[ds][
        transformed_mtb[ds]['Sample'].isin(samples)].reset_index(drop=True)
    return spc_sub, mtb_sub


_counts = (harmonized_meta[CRC_DS_HUB].groupby('Stage.Group')['Sample'].count()
           .reindex(STAGE_GROUP_ORDER))
print('\nSamples per Stage.Group (primary set only):')
print(_counts.to_string())
print(f'\nTotal primary: {int(_counts.sum())}')
print('Group sizes differ, so edge counts below are power-limited -- see the caveat above.')

In [ ]:
# Compute per-group polyamine-centric correlation networks
stratified_networks = {}
pa_targets_hub = list(polyamine_columns.get(CRC_DS_HUB, {}).values())
MIN_SAMPLES = 15

for group_name in STAGE_GROUP_ORDER:
    spc_sub, mtb_sub = filter_by_group(CRC_DS_HUB, group_name)
    n_samples = len(spc_sub)
    print(f'\n{group_name}: {n_samples} samples')

    if n_samples < MIN_SAMPLES:
        print(f'  Skipping -- fewer than {MIN_SAMPLES} samples')
        stratified_networks[group_name] = nx.Graph()
        continue

    targets = pa_targets_hub if pa_targets_hub else None
    corr_df = compute_correlations(spc_sub, mtb_sub, target_metabolites=targets)

    G = build_correlation_network(corr_df, rho_threshold=0.3, q_threshold=0.05)
    stratified_networks[group_name] = G

    n_spc = sum(1 for _, d in G.nodes(data=True) if d.get('node_type') == 'species')
    n_mtb = sum(1 for _, d in G.nodes(data=True) if d.get('node_type') == 'metabolite')
    print(f'  Network: {G.number_of_nodes()} nodes '
          f'({n_spc} species, {n_mtb} metabolites), '
          f'{G.number_of_edges()} edges')

In [ ]:
# Extract hub species per group & save CSV
pa_node_set_hub = set(polyamine_columns.get(CRC_DS_HUB, {}).values())
stratified_hub_rows = []

for group_name, G in stratified_networks.items():
    species_nodes = [n for n in G.nodes() if G.nodes[n].get('node_type') == 'species']
    for spc in species_nodes:
        pa_neighbors = [nb for nb in G.neighbors(spc) if nb in pa_node_set_hub]
        if pa_neighbors:
            stratified_hub_rows.append({
                'Group':                   group_name,
                'Species':                 spc,
                'N_polyamine_connections': len(pa_neighbors),
                'Degree':                  G.degree(spc),
                'Connected_polyamines':    ', '.join(pa_neighbors),
            })

stratified_hub_df = pd.DataFrame(stratified_hub_rows)
if not stratified_hub_df.empty:
    stratified_hub_df = stratified_hub_df.sort_values(
        ['Group', 'N_polyamine_connections'], ascending=[True, False]
    )
stratified_hub_df.to_csv(
    RESULTS_DIR / 'tables' / 'stratified_hub_species_v2.csv', index=False
)
print('Hub species per group:')
if not stratified_hub_df.empty:
    print(stratified_hub_df.groupby('Group').size().reindex(STAGE_GROUP_ORDER, fill_value=0).to_string())
    display(stratified_hub_df.head(20))
else:
    print('  None found.')

In [ ]:
# Healthy hub species network -- single figure
import matplotlib.patches as mpatches

_GROUP = 'Healthy'
G_h    = stratified_networks.get(_GROUP, nx.Graph())

fig, ax = plt.subplots(figsize=(14, 10))

if G_h.number_of_nodes() > 0:
    pos = nx.spring_layout(G_h, k=2.0, seed=42)

    node_colors, node_sizes = [], []
    for n in G_h.nodes():
        if n in pa_node_set_hub:
            node_colors.append('#e74c3c'); node_sizes.append(600)
        elif G_h.nodes[n].get('node_type') == 'species':
            node_colors.append('#3498db'); node_sizes.append(220)
        else:
            node_colors.append('#2ecc71'); node_sizes.append(160)

    edge_colors = [
        '#e74c3c' if G_h[u][v].get('sign') == 'negative' else '#27ae60'
        for u, v in G_h.edges()
    ]
    nx.draw(G_h, pos, node_color=node_colors, node_size=node_sizes,
            edge_color=edge_colors, alpha=0.7, ax=ax, with_labels=False,
            width=0.8)

    labels = {}
    for n in G_h.nodes():
        if n in pa_node_set_hub:
            labels[n] = (n.split('_', 1)[-1] if '_' in n else n)[:30]
        elif G_h.nodes[n].get('node_type') == 'species' and G_h.degree(n) >= 3:
            labels[n] = n[:28]
    nx.draw_networkx_labels(G_h, pos, labels, font_size=7, ax=ax)

    legend_patches = [
        mpatches.Patch(color='#e74c3c', label='Polyamine'),
        mpatches.Patch(color='#3498db', label='Species'),
        mpatches.Patch(color='#2ecc71', label='Other metabolite'),
        mpatches.Patch(color='#e74c3c', label='Negative corr.', alpha=0.5),
        mpatches.Patch(color='#27ae60', label='Positive corr.', alpha=0.5),
    ]
    ax.legend(handles=legend_patches, loc='upper left', fontsize=9, framealpha=0.85)
    ax.set_title(
        f'Hub Species Network -- Healthy (YACHIDA-CRC-2019)\n'
        f'{G_h.number_of_nodes()} nodes | {G_h.number_of_edges()} edges',
        fontsize=13, fontweight='bold'
    )
else:
    ax.text(0.5, 0.5, 'No significant correlations found',
            ha='center', va='center', transform=ax.transAxes, fontsize=13)
    ax.set_title('Hub Species Network -- Healthy', fontsize=13, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
out_healthy = RESULTS_DIR / 'figures' / 'network' / 'hub_network_Healthy_v2.png'
plt.savefig(out_healthy, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'Saved: {out_healthy}')

In [ ]:
# Non-Healthy multi-panel hub species networks (Early CRC | Advanced CRC)
from matplotlib.lines import Line2D

NON_HEALTHY = [g for g in STAGE_GROUP_ORDER if g != 'Healthy']   # Early_CRC, Advanced_CRC
PANEL_TITLES = {
    'Early_CRC':    'Early CRC (Stage 0 + I/II)',
    'Advanced_CRC': 'Advanced CRC (Stage III/IV)',
}

fig, axes = plt.subplots(1, len(NON_HEALTHY), figsize=(14 * len(NON_HEALTHY), 14),
                          squeeze=False)
axes = axes.flatten()
fig.suptitle(
    'Non-Healthy Hub Species Networks by CRC Stage Group\n'
    '(YACHIDA-CRC-2019, primary set; MP excluded from the study, PostSurgery off-axis)\n'
    'Edge counts are power-limited by group size and are not interpreted as biology',
    fontsize=15, fontweight='bold', y=1.02
)

for ax, group_name in zip(axes, NON_HEALTHY):
    G_g   = stratified_networks.get(group_name, nx.Graph())
    n_grp = int((harmonized_meta[CRC_DS_HUB]['Stage.Group'] == group_name).sum())
    title = f"{PANEL_TITLES[group_name]}  (n={n_grp})"

    if G_g.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'No significant\ncorrelations found',
                ha='center', va='center', transform=ax.transAxes, fontsize=13)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        continue

    pos = nx.spring_layout(G_g, k=2.0, seed=42)

    node_colors, node_sizes = [], []
    for n in G_g.nodes():
        if n in pa_node_set_hub:
            node_colors.append('#e74c3c'); node_sizes.append(600)
        elif G_g.nodes[n].get('node_type') == 'species':
            node_colors.append('#3498db'); node_sizes.append(220)
        else:
            node_colors.append('#2ecc71'); node_sizes.append(160)

    edge_colors = ['#e74c3c' if G_g[u][v].get('sign') == 'negative' else '#27ae60'
                   for u, v in G_g.edges()]
    nx.draw(G_g, pos, node_color=node_colors, node_size=node_sizes,
            edge_color=edge_colors, alpha=0.7, ax=ax, with_labels=False, width=0.8)

    labels = {}
    for n in G_g.nodes():
        if n in pa_node_set_hub:
            labels[n] = (n.split('_', 1)[-1] if '_' in n else n)[:28]
        elif G_g.nodes[n].get('node_type') == 'species' and G_g.degree(n) >= 3:
            labels[n] = n[:26]
    nx.draw_networkx_labels(G_g, pos, labels, font_size=6, ax=ax)

    n_spc = sum(1 for _, d in G_g.nodes(data=True) if d.get('node_type') == 'species')
    ax.set_title(f'{title}\n{G_g.number_of_nodes()} nodes ({n_spc} spc) | '
                 f'{G_g.number_of_edges()} edges', fontsize=13, fontweight='bold')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=14,
           label='Polyamine'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=14,
           label='Species'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=12,
           label='Other metabolite'),
    Line2D([0], [0], color='#e74c3c', lw=2.5, label='Negative correlation'),
    Line2D([0], [0], color='#27ae60', lw=2.5, label='Positive correlation'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=12,
           bbox_to_anchor=(0.5, -0.04), framealpha=0.9)

plt.tight_layout()
out_multi = RESULTS_DIR / 'figures' / 'network' / 'hub_network_non_healthy_multifig_v2.png'
plt.savefig(out_multi, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved: {out_multi}')

---
## Step 10: CRC Progression Analysis

**Scope: Yachida only** (Sinha and Kim are binary Healthy/CRC cohorts with no staging), and
within Yachida the **primary** set only.

Two complementary approaches:
- **10A — ordinal Spearman**: correlate species and metabolites with ordered stage
  (`utils.CRC_STAGE_NUMERIC`: Healthy=0, Stage_0=1, Stage_I_II=2, Stage_III_IV=3).
  `HS`/`PostSurgery` is **NaN** on this scale by construction, so it cannot enter the trend
  test even if a caller forgets to filter — and it has already been removed from these
  matrices anyway. MP does not exist in the data.
- **10B — binary early vs late**: `Stage_I_II` vs `Stage_III_IV`, excluding Healthy and
  Stage_0, giving a direct test of what separates early from advanced disease.

### 8.5 Stage-group dashboard

Cohort composition and hub-network size across the three primary `Stage.Group` tiers, in
clinical order (`Healthy -> Early_CRC -> Advanced_CRC`). The left panel uses a single-hue
sequential ramp stepped by severity rather than unrelated categorical colours, because these
are an *ordered* progression; the darkening blue reinforces the same ordering
`utils.STAGE_GROUP_ORDER` encodes everywhere else.

The right panel repeats the caveat that matters: network size tracks group size here, so the
two series are shown together with n annotated and are not read as a biological trend.

In [ ]:
# Stage-group dashboard: sample composition (left) + hub-network size (right).
# Sequential blue ramp for the ordered severity tiers (never unrelated categorical
# hues for what is one ordinal axis); categorical blue/orange for the two
# independent network-size series on the right.
STAGE_RAMP = ['#9ec5f4', '#2a78d6', '#104281']          # one step per primary tier
SERIES_BLUE, SERIES_ORANGE = '#2a78d6', '#eb6834'
INK, INK_SECONDARY, MUTED, GRID = '#0b0b0b', '#52514e', '#898781', '#e1e0d9'

_counts_dash = (harmonized_meta[CRC_DS_HUB]
                .groupby('Stage.Group')['Sample'].count()
                .reindex(STAGE_GROUP_ORDER, fill_value=0))
_net_sizes = [
    (g, stratified_networks.get(g, nx.Graph()).number_of_nodes(),
        stratified_networks.get(g, nx.Graph()).number_of_edges())
    for g in STAGE_GROUP_ORDER
]

fig, (ax_n, ax_g) = plt.subplots(1, 2, figsize=(13, 4.6))
fig.patch.set_facecolor('#fcfcfb')

# -- left: sample N per stage group, sequential-by-severity --
y_pos = range(len(STAGE_GROUP_ORDER))
ax_n.barh(y_pos, _counts_dash.values, color=STAGE_RAMP, height=0.62,
          edgecolor='white', linewidth=0.6)
for i, v in enumerate(_counts_dash.values):
    ax_n.text(v + max(_counts_dash.values) * 0.015, i, f'n={v}',
              va='center', fontsize=10, color=INK)
ax_n.set_yticks(list(y_pos))
ax_n.set_yticklabels(STAGE_GROUP_ORDER, fontsize=10.5, color=INK)
ax_n.invert_yaxis()
ax_n.set_xlabel('Samples', fontsize=10.5, color=INK_SECONDARY)
ax_n.set_title(f'Cohort composition (primary set, n={int(_counts_dash.sum())})',
               fontsize=12, fontweight='bold', color=INK, loc='left')
for spine in ['top', 'right']:
    ax_n.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax_n.spines[spine].set_color(GRID)
ax_n.tick_params(colors=MUTED)
ax_n.set_facecolor('#fcfcfb')

# -- right: hub-network nodes vs edges per stage group --
labels_g = [g for g, _, _ in _net_sizes]
nodes_g  = [n for _, n, _ in _net_sizes]
edges_g  = [e for _, _, e in _net_sizes]
x = range(len(labels_g))
w = 0.36
ax_g.bar([i - w/2 for i in x], nodes_g, width=w, color=SERIES_BLUE,
         label='Nodes', edgecolor='white', linewidth=0.6)
ax_g.bar([i + w/2 for i in x], edges_g, width=w, color=SERIES_ORANGE,
         label='Edges', edgecolor='white', linewidth=0.6)
_ymax = max(edges_g + nodes_g + [1])
for i, (n, e) in enumerate(zip(nodes_g, edges_g)):
    if n: ax_g.text(i - w/2, n + _ymax * 0.02, str(n), ha='center', fontsize=9, color=INK_SECONDARY)
    if e: ax_g.text(i + w/2, e + _ymax * 0.02, str(e), ha='center', fontsize=9, color=INK_SECONDARY)
ax_g.set_xticks(list(x))
ax_g.set_xticklabels([f'{g}\n(n={_counts_dash[g]})' for g in labels_g],
                     fontsize=9, color=INK)
ax_g.set_ylabel('Count', fontsize=10.5, color=INK_SECONDARY)
ax_g.set_title('Hub-network size — tracks group size, not biology',
               fontsize=12, fontweight='bold', color=INK, loc='left')
ax_g.legend(frameon=False, fontsize=9.5, loc='upper right')
for spine in ['top', 'right']:
    ax_g.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax_g.spines[spine].set_color(GRID)
ax_g.tick_params(colors=MUTED)
ax_g.set_facecolor('#fcfcfb')
ax_g.grid(axis='y', color=GRID, linewidth=0.7, zorder=0)
ax_g.set_axisbelow(True)

plt.tight_layout()
out_dash = RESULTS_DIR / 'figures' / 'network' / 'stage_group_dashboard_v2.png'
plt.savefig(out_dash, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show(); plt.close()
print(f'Saved: {out_dash}')

In [ ]:
CRC_DS = DATASET_DISCOVERY

print('Yachida stage distribution (primary set):')
print(harmonized_meta[CRC_DS]['Study.Group'].value_counts()
      .reindex(CRC_STAGE_ORDER, fill_value=0).to_string())

print('\nOrdinal stage coding actually in use (utils.CRC_STAGE_NUMERIC):')
print(harmonized_meta[CRC_DS][['Study.Group', 'CRC.Stage.Numeric']]
      .drop_duplicates().sort_values('CRC.Stage.Numeric').to_string(index=False))

# The trend tests below depend on this: every primary sample must have a finite
# ordinal position, and the sensitivity group must be absent.
assert harmonized_meta[CRC_DS]['CRC.Stage.Numeric'].notna().all(), \
    'a primary sample has no ordinal stage'
assert not harmonized_meta[CRC_DS]['Stage.Group'].eq('PostSurgery').any()
print(f'\nAll {len(harmonized_meta[CRC_DS])} primary samples carry a finite ordinal stage.')

### 10A: Ordinal Spearman Correlation with CRC Stage

For each species x metabolite pair this computes:
- `Rho_species_stage`: how strongly does species abundance track CRC progression?
- `Rho_metabolite_stage`: how strongly does metabolite abundance track progression?
- `Rho_spc_mtb`: species-metabolite co-variation

Pairs that are significant in all three tests are the strongest candidate
producer-metabolite associations driven by CRC progression.

**Fix applied:** `utils.progression_correlation_analysis()` already fixes the old FDR
bug internally -- it computes each unique species-stage and metabolite-stage
correlation **once**, FDR-corrects on that deduplicated set of true tests, and only
then joins the corrected q-values back onto the long species x metabolite pairwise
table. The old notebook instead ran `multipletests()` directly on the full long-format
table, where every unique species-stage p-value appeared once per metabolite partner
(and vice versa) -- with ~500 species x ~156 metabolites = ~78,000 rows but only
~500 + 156 = ~656 genuinely unique marginal tests, that inflated the effective test
count by roughly two orders of magnitude and corrupted every reported marginal q-value.
This notebook
just calls the already-fixed function -- no correlation/FDR logic is reimplemented here
-- and reports significant species/metabolite counts via `.unique()` on the
deduplicated marginal columns, not `.sum()` over the long pairwise table.

In [ ]:
print('Running CRC progression correlation analysis (ordinal stage)...')
prog_corr = progression_correlation_analysis(
    transformed_species[CRC_DS],
    transformed_mtb[CRC_DS],
    harmonized_meta[CRC_DS],
    stage_col='CRC.Stage.Numeric',
    min_samples=10,
)

if not prog_corr.empty:
    print(f'\nResults: {len(prog_corr)} species x metabolite pairs computed')

    # IMPORTANT: species/metabolite significance counts use .unique() -- each
    # unique species (or metabolite) appears in many pairwise rows, so a naive
    # .sum() over QVal_species_stage < 0.05 rows massively over-counts how many
    # DISTINCT species/metabolites are actually significant (this was exactly
    # the old notebook's bug -- see markdown above).
    sig_species     = prog_corr.loc[prog_corr['QVal_species_stage'] < 0.05, 'Species'].unique()
    sig_metabolites = prog_corr.loc[prog_corr['QVal_metabolite_stage'] < 0.05, 'Metabolite'].unique()
    n_sig_both  = ((prog_corr['QVal_species_stage'] < 0.05) &
                   (prog_corr['QVal_metabolite_stage'] < 0.05)).sum()
    n_sig_all3  = ((prog_corr['QVal_species_stage'] < 0.05) &
                   (prog_corr['QVal_metabolite_stage'] < 0.05) &
                   (prog_corr['QVal_spc_mtb'] < 0.05)).sum()

    print(f'  Unique species significantly correlated with stage (q<0.05):     {len(sig_species)}')
    print(f'  Unique metabolites significantly correlated with stage (q<0.05): {len(sig_metabolites)}')
    print(f'  Pairs where BOTH species AND metabolite track stage:             {n_sig_both}')
    print(f'  Pairs where all 3 tests significant (strongest candidates):     {n_sig_all3}')

    prog_corr.to_csv(
        RESULTS_DIR / 'tables' / 'CRC_progression_ordinal_correlations_v2.csv', index=False)
else:
    print('No progression correlations computed -- check stage data.')

In [ ]:
# Top progression-linked species-metabolite pairs (all 3 tests significant)
all3_sig = pd.DataFrame()
if not prog_corr.empty:
    all3_sig = prog_corr[
        (prog_corr['QVal_species_stage']    < 0.05) &
        (prog_corr['QVal_metabolite_stage'] < 0.05) &
        (prog_corr['QVal_spc_mtb']          < 0.05)
    ].copy()

    print('=== Top pairs (all 3 tests q<0.05) ===')
    print('Pairs where species tracks stage, metabolite tracks stage, '
          'AND species correlates with metabolite:')

    if not all3_sig.empty:
        all3_sig['combined_effect'] = (
            all3_sig['Rho_species_stage'].abs() +
            all3_sig['Rho_metabolite_stage'].abs() +
            all3_sig['Rho_spc_mtb'].abs()
        )
        top30 = all3_sig.sort_values('combined_effect', ascending=False).head(30)
        display(top30[['Species', 'Metabolite',
                        'Rho_species_stage', 'QVal_species_stage',
                        'Rho_metabolite_stage', 'QVal_metabolite_stage',
                        'Rho_spc_mtb', 'QVal_spc_mtb',
                        'combined_effect']].round(3))
        top30.to_csv(
            RESULTS_DIR / 'tables' / 'CRC_progression_top_pairs_v2.csv', index=False)

        pa_cols_set = set()
        for ds in POLYAMINE_DATASETS_ANALYSIS:
            if ds in polyamine_columns:
                pa_cols_set.update(polyamine_columns[ds].values())
        pa_progression = all3_sig[all3_sig['Metabolite'].isin(pa_cols_set)]
        if not pa_progression.empty:
            print(f'\nPolyamine-specific progression pairs: {len(pa_progression)}')
            display(pa_progression[['Species', 'Metabolite',
                                     'Rho_species_stage', 'Rho_metabolite_stage',
                                     'Rho_spc_mtb']].round(3))
    else:
        print('No pairs pass all 3 tests at q<0.05. Showing top by species-stage Rho:')
        display(prog_corr.head(20)[[
            'Species', 'Metabolite',
            'Rho_species_stage', 'QVal_species_stage',
            'Rho_metabolite_stage', 'QVal_metabolite_stage',
            'Rho_spc_mtb', 'QVal_spc_mtb',
        ]].round(3))

In [ ]:
# Volcano-style scatter: Rho_species_stage vs Rho_metabolite_stage,
# Sequential blue ramp (validated design-system palette) instead of an
# ad hoc YlOrRd colormap -- this is a magnitude encoding (|Rho|), so it
# gets one hue, light->dark, like every other sequential encoding in this
# notebook's figures.
from matplotlib.colors import LinearSegmentedColormap
_seq_blue = LinearSegmentedColormap.from_list(
    'seq_blue', ['#cde2fb', '#6da7ec', '#2a78d6', '#104281'])
# coloured by |Rho_spc_mtb|, annotating polyamines
if not prog_corr.empty:
    pa_cols_set = set()
    for ds in POLYAMINE_DATASETS_ANALYSIS:
        if ds in polyamine_columns:
            pa_cols_set.update(polyamine_columns[ds].values())

    fig, ax = plt.subplots(figsize=(10, 8))

    scatter = ax.scatter(
        prog_corr['Rho_species_stage'],
        prog_corr['Rho_metabolite_stage'],
        c=prog_corr['Rho_spc_mtb'].abs(),
        cmap=_seq_blue, alpha=0.45, s=12, vmin=0, vmax=0.6,
    )
    plt.colorbar(scatter, ax=ax, label='|Rho species-metabolite|')

    pa_sub = prog_corr[prog_corr['Metabolite'].isin(pa_cols_set)]
    ax.scatter(pa_sub['Rho_species_stage'], pa_sub['Rho_metabolite_stage'],
               color='#e87ba4', s=60, zorder=5, label='Polyamine metabolites')

    if not all3_sig.empty:
        for _, row in all3_sig.sort_values('combined_effect', ascending=False).head(10).iterrows():
            ax.annotate(
                f"{row['Species'][:20]}\n->{row['Metabolite'][:20]}",
                (row['Rho_species_stage'], row['Rho_metabolite_stage']),
                fontsize=6, alpha=0.8
            )

    ax.axhline(0, color='grey', ls='--', alpha=0.4)
    ax.axvline(0, color='grey', ls='--', alpha=0.4)
    ax.set_xlabel('Spearman Rho: Species ~ CRC Stage')
    ax.set_ylabel('Spearman Rho: Metabolite ~ CRC Stage')
    ax.set_title('CRC Progression: Species x Metabolite Ordinal Correlations\n'
                 'Quadrant I (top-right) = both increase with stage', fontsize=11)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures' / 'progression' / 'ordinal_progression_scatter_v2.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Stage-trend box plots for top progression pairs
os.makedirs(RESULTS_DIR / 'figures' / 'progression', exist_ok=True)

if not prog_corr.empty:
    plot_pairs_source = all3_sig if not all3_sig.empty else prog_corr
    sort_col = 'combined_effect' if 'combined_effect' in plot_pairs_source.columns else 'QVal_species_stage'
    ascending = sort_col != 'combined_effect'
    top_plot = plot_pairs_source.sort_values(sort_col, ascending=ascending).head(10)

    for _, row in top_plot.iterrows():
        spc_name = row['Species']
        mtb_name = row['Metabolite']
        if (spc_name not in transformed_species[CRC_DS].columns or
                mtb_name not in transformed_mtb[CRC_DS].columns):
            continue
        safe_name = f"{spc_name[:20]}_{mtb_name[:20]}".replace(' ', '_').replace('/', '_')
        fig, axes = plot_stage_trend(
            spc_name, mtb_name,
            transformed_species[CRC_DS],
            transformed_mtb[CRC_DS],
            harmonized_meta[CRC_DS],
            save_path=RESULTS_DIR / 'figures' / 'progression' / f'trend_{safe_name}_v2.png'
        )
        plt.show()
        plt.close()

### 10B: Binary Early vs Late CRC

Comparison: `Stage_I_II` vs `Stage_III_IV`. Healthy and Stage_0 samples are excluded, so
this is a within-cancer contrast: what separates early-invasive from advanced disease.
MP is not in the study and PostSurgery is not in these matrices.

Note this is a *different* contrast from Step 6B's `Early_CRC vs Advanced_CRC`, which pools
Stage_0 into the early arm. Both are reported because they answer slightly different
questions, and the stage-group version is the one NB03 draws targets from.

Tumour location is a legitimate covariate for a contrast like this one (both arms are
carcinomas) — that analysis is NB04's, which adjusts for site and reports
rectum/left/right subgroups. It is not repeated here.

In [ ]:
# Subset Yachida to Stage_I_II and Stage_III_IV only
meta_crc = harmonized_meta[CRC_DS]
crc_samples = meta_crc[
    meta_crc['Study.Group'].isin(['Stage_I_II', 'Stage_III_IV'])
]['Sample'].tolist()

meta_binary = meta_crc[meta_crc['Sample'].isin(crc_samples)].copy()
spc_binary  = transformed_species[CRC_DS][
    transformed_species[CRC_DS]['Sample'].isin(crc_samples)].copy()
mtb_binary  = transformed_mtb[CRC_DS][
    transformed_mtb[CRC_DS]['Sample'].isin(crc_samples)].copy()

print(f'Binary CRC subset: {len(crc_samples)} samples')
print(meta_binary['Study.Group'].value_counts().to_string())

In [ ]:
# Differential abundance: Stage_I_II vs Stage_III_IV
print('Differential abundance: Stage_I_II vs Stage_III_IV')

da_early_late_spc = differential_abundance(
    spc_binary, meta_binary, 'Stage_I_II', 'Stage_III_IV')
da_early_late_mtb = differential_abundance(
    mtb_binary, meta_binary, 'Stage_I_II', 'Stage_III_IV')

n_sig_spc = da_early_late_spc['Significant'].sum() if not da_early_late_spc.empty else 0
n_sig_mtb = da_early_late_mtb['Significant'].sum() if not da_early_late_mtb.empty else 0
print(f'  Species significantly different (q<0.05):     {n_sig_spc}')
print(f'  Metabolites significantly different (q<0.05): {n_sig_mtb}')

da_early_late_spc.to_csv(
    RESULTS_DIR / 'tables' / 'CRC_StageI_II_vs_III_IV_species_DA_v2.csv', index=False)
da_early_late_mtb.to_csv(
    RESULTS_DIR / 'tables' / 'CRC_StageI_II_vs_III_IV_metabolites_DA_v2.csv', index=False)

In [ ]:
# Volcano plots
pa_features_crc = list(polyamine_columns[CRC_DS].values())

fig, ax = volcano_plot(
    da_early_late_mtb,
    title='Yachida -- Metabolites: Stage_I_II vs Stage_III_IV',
    fc_col='MeanDiff',
    highlight_features=pa_features_crc,
    save_path=RESULTS_DIR / 'figures' / 'progression' / 'binary_StageI_II_vs_III_IV_mtb_volcano_v2.png'
)
plt.show()
plt.close()

fig, ax = volcano_plot(
    da_early_late_spc,
    title='Yachida -- Species: Stage_I_II vs Stage_III_IV',
    fc_col='MeanDiff',
    save_path=RESULTS_DIR / 'figures' / 'progression' / 'binary_StageI_II_vs_III_IV_spc_volcano_v2.png'
)
plt.show()
plt.close()

In [ ]:
# Species-metabolite correlations in binary CRC subset
print('Species-metabolite correlations in Stage_I_II + Stage_III_IV samples...')

pa_cols_crc = list(polyamine_columns[CRC_DS].values())
spc_cols_b  = [c for c in spc_binary.columns if c != 'Sample']
mtb_cols_b  = [c for c in mtb_binary.columns if c != 'Sample']

if len(spc_cols_b) > MAX_SPC_FEATURES:
    spc_top_var = (spc_binary[spc_cols_b].var()
                   .nlargest(MAX_SPC_FEATURES).index.tolist())
    spc_df_b = spc_binary[['Sample'] + spc_top_var]
else:
    spc_df_b = spc_binary

if len(mtb_cols_b) > MAX_MTB_FEATURES:
    mtb_top_var = (mtb_binary[mtb_cols_b].var()
                   .nlargest(MAX_MTB_FEATURES).index.tolist())
    for pa_col in pa_cols_crc:
        if pa_col not in mtb_top_var:
            mtb_top_var.append(pa_col)
    mtb_df_b = mtb_binary[['Sample'] + mtb_top_var]
else:
    mtb_df_b = mtb_binary

corr_binary = compute_correlations(spc_df_b, mtb_df_b)
n_sig_b = (corr_binary['QValue'] < 0.05).sum() if not corr_binary.empty else 0
print(f'  {len(corr_binary)} pairs tested, {n_sig_b} significant (q<0.05)')
corr_binary.to_csv(
    RESULTS_DIR / 'tables' / 'CRC_binary_stage_correlations_v2.csv', index=False)

In [ ]:
# Overlap: species/metabolites significant in both 10A (ordinal) and 10B (binary)
if not da_early_late_spc.empty and not prog_corr.empty:
    sig_binary_spc  = set(da_early_late_spc[da_early_late_spc['Significant']]['Feature'])
    sig_ordinal_spc = set(prog_corr[prog_corr['QVal_species_stage'] < 0.05]['Species'].unique())
    overlap_spc     = sig_binary_spc & sig_ordinal_spc

    sig_binary_mtb  = set(da_early_late_mtb[da_early_late_mtb['Significant']]['Feature'])
    sig_ordinal_mtb = set(prog_corr[prog_corr['QVal_metabolite_stage'] < 0.05]['Metabolite'].unique())
    overlap_mtb     = sig_binary_mtb & sig_ordinal_mtb

    print('=' * 70)
    print('OVERLAP: Features significant in BOTH ordinal (10A) and binary (10B)')
    print('=' * 70)
    print(f'Species in both:     {len(overlap_spc)}')
    print(f'Metabolites in both: {len(overlap_mtb)}')

    if overlap_spc:
        print('\nOverlap species (top 10 by binary q-value):')
        ovl_spc_df = da_early_late_spc[
            da_early_late_spc['Feature'].isin(overlap_spc)
        ].sort_values('QValue').head(10)
        display(ovl_spc_df[['Feature', 'MeanDiff', 'QValue', 'CliffsD']].round(3))

    if overlap_mtb:
        print('\nOverlap metabolites (highlight polyamines):')
        pa_cols_set_overlap = set()
        for ds in POLYAMINE_DATASETS_ANALYSIS:
            if ds in polyamine_columns:
                pa_cols_set_overlap.update(polyamine_columns[ds].values())
        ovl_mtb_df = da_early_late_mtb[
            da_early_late_mtb['Feature'].isin(overlap_mtb)
        ].sort_values('QValue').copy()
        ovl_mtb_df['IsPolyamine'] = ovl_mtb_df['Feature'].isin(pa_cols_set_overlap)
        display(ovl_mtb_df[['Feature', 'MeanDiff', 'QValue', 'IsPolyamine']].round(3))

    pd.DataFrame({'Species_overlap': sorted(overlap_spc)}).to_csv(
        RESULTS_DIR / 'tables' / 'CRC_progression_species_overlap_10A_10B_v2.csv', index=False)
    pd.DataFrame({'Metabolite_overlap': sorted(overlap_mtb)}).to_csv(
        RESULTS_DIR / 'tables' / 'CRC_progression_metabolite_overlap_10A_10B_v2.csv', index=False)

---
## Save Analysis Results

Writes `analysis_results_v2.pkl` with the keys required downstream: `da_results`,
`polyamine_corr`, `full_corr`, `consistent_pairs`, `network_metrics`, `prog_corr_ordinal`,
`da_group_da_spc`, `da_group_da_mtb`, `partial_corr`, `genus_polyamine_corr`,
`pa_match_map`, `proxy_used_by_ds`, `robust_summary`.

10B's binary early-vs-late results and the within-subset correlation table stay as CSVs
rather than being duplicated into this pickle, so the pickle's contents match the NB02
contract exactly.

### 11. Results dashboard

A single consolidated view of every differential-abundance comparison run in
this notebook (fine-grained `Study.Group` scheme + the grouped `Stage.Group`
scheme), replacing a page of scattered print statements with one chart.
Species and metabolites are fixed categorical slots 1/2 (blue/orange) across
every panel of this notebook -- the same two colors mean the same two things
everywhere they appear.

In [ ]:
# Consolidated differential-abundance dashboard: one horizontal bar per
# comparison, species vs metabolites as fixed categorical slots (blue/orange),
# fine-grained and grouped schemes in separate facets since they're different
# sample universes, not directly comparable bar-for-bar.
SERIES_BLUE, SERIES_ORANGE = '#2a78d6', '#eb6834'
INK, INK_SECONDARY, MUTED, GRID = '#0b0b0b', '#52514e', '#898781', '#e1e0d9'

_fine_rows = []
for ds in DATASETS_ANALYSIS:
    for (g1, g2), results in da_results[ds].items():
        n_spc = int(results['species']['Significant'].sum()) if not results['species'].empty else 0
        n_mtb = int(results['metabolites']['Significant'].sum()) if not results['metabolites'].empty else 0
        _fine_rows.append({'Label': f'{ds.split("-")[0].split("_")[0]}: {g1} vs {g2}',
                            'Species': n_spc, 'Metabolites': n_mtb})

_group_rows = []
for (g1, g2), spc_da in da_group_da_spc.items():
    mtb_da = da_group_da_mtb[(g1, g2)]
    n_spc = int(spc_da['Significant'].sum()) if not spc_da.empty else 0
    n_mtb = int(mtb_da['Significant'].sum()) if not mtb_da.empty else 0
    _group_rows.append({'Label': f'{g1} vs {g2}', 'Species': n_spc, 'Metabolites': n_mtb})

_fine_df  = pd.DataFrame(_fine_rows)
_group_df = pd.DataFrame(_group_rows)

fig, (ax_f, ax_g2) = plt.subplots(
    2, 1, figsize=(11, 0.62 * (len(_fine_df) + len(_group_df)) + 2.6),
    gridspec_kw={'height_ratios': [len(_fine_df), len(_group_df)]},
)
fig.patch.set_facecolor('#fcfcfb')

for ax, df, subtitle in [(ax_f, _fine_df, 'Fine-grained Study.Group scheme'),
                          (ax_g2, _group_df, 'Grouped Stage.Group scheme')]:
    y = range(len(df))
    h = 0.34
    ax.barh([i + h/2 for i in y], df['Species'], height=h, color=SERIES_BLUE,
            label='Species', edgecolor='white', linewidth=0.6)
    ax.barh([i - h/2 for i in y], df['Metabolites'], height=h, color=SERIES_ORANGE,
            label='Metabolites', edgecolor='white', linewidth=0.6)
    xmax = max(df['Species'].max(), df['Metabolites'].max(), 1)
    for i, (s, m) in enumerate(zip(df['Species'], df['Metabolites'])):
        if s: ax.text(s + xmax * 0.015, i + h/2, str(s), va='center', fontsize=9, color=INK_SECONDARY)
        if m: ax.text(m + xmax * 0.015, i - h/2, str(m), va='center', fontsize=9, color=INK_SECONDARY)
    ax.set_yticks(list(y))
    ax.set_yticklabels(df['Label'], fontsize=9.5, color=INK)
    ax.invert_yaxis()
    ax.set_xlabel('Significant features (q<0.05)', fontsize=10, color=INK_SECONDARY)
    ax.set_title(subtitle, fontsize=11.5, fontweight='bold', color=INK, loc='left')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color(GRID)
    ax.tick_params(colors=MUTED)
    ax.set_facecolor('#fcfcfb')

ax_f.legend(frameon=False, fontsize=9.5, loc='lower right')
fig.suptitle('Differential Abundance -- Results Dashboard', fontsize=13.5,
             fontweight='bold', color=INK, x=0.02, ha='left', y=1.0)
plt.tight_layout(rect=[0, 0, 1, 0.97])
out_dash_a = RESULTS_DIR / 'figures' / 'differential' / 'da_results_dashboard_v2.png'
plt.savefig(out_dash_a, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
plt.close()
print(f'Saved: {out_dash_a}')


In [ ]:
analysis_results = {
    'da_results':           da_results,
    'polyamine_corr':       polyamine_corr,
    'full_corr':            full_corr,
    'consistent_pairs':     consistent_pairs if not consistent_pairs.empty else consistent_relaxed,
    'consistent_threshold': 'q<0.05' if not consistent_pairs.empty else 'q<0.10 (exploratory)',
    'network_metrics':      network_metrics,
    'prog_corr_ordinal':    prog_corr,
    'da_group_da_spc':      da_group_da_spc,
    'da_group_da_mtb':      da_group_da_mtb,
    'partial_corr':         partial_corr_results,
    'genus_polyamine_corr': genus_polyamine_corr,
    'pa_match_map':         pa_match_map,
    'proxy_used_by_ds':     proxy_used_by_ds,
    'robust_summary':       robust_df,
    'da_covariates':        DA_COVARS,
    'sample_scope':         'primary (utils.primary_mask)',
}

with open(RESULTS_DIR / 'intermediate' / 'analysis_results_v2.pkl', 'wb') as f:
    pickle.dump(analysis_results, f, protocol=pickle.HIGHEST_PROTOCOL)

print('Analysis results saved to analysis_results_v2.pkl')
print('Keys:', list(analysis_results.keys()))

In [ ]:
print('\n' + '=' * 80)
print('ANALYSIS COMPLETE -- SUMMARY (v2.1, primary analysis set)')
print('=' * 80)

print(f'\nSample scope: primary set only (utils.primary_mask). '
      f'MP excluded from the study; PostSurgery held aside.')
for ds in DATASETS_ANALYSIS:
    print(f'  {ds}: n={len(harmonized_meta[ds])}')

print('\n1. Differential Abundance (fine-grained Study.Group scheme, unadjusted):')
for ds in DATASETS_ANALYSIS:
    for (g1, g2), results in da_results[ds].items():
        n_spc = results['species']['Significant'].sum()     if not results['species'].empty     else 0
        n_mtb = results['metabolites']['Significant'].sum() if not results['metabolites'].empty else 0
        print(f'  {ds} ({g1} vs {g2}): {n_spc} species, {n_mtb} metabolites sig.')

print('\n1B. Differential Abundance (grouped scheme, BOTH tests -- robust = both + same direction):')
print(robust_df.to_string(index=False))

print('\n2. Polyamine correlations (species level, per cohort):')
for ds in POLYAMINE_DATASETS_ANALYSIS:
    if ds in polyamine_corr and not polyamine_corr[ds].empty:
        n_sig = int((polyamine_corr[ds]['QValue'] < 0.05).sum())
        print(f'  {ds}: {n_sig} significant species-polyamine pairs (q<0.05)')

print('\n2B. Partial correlations (adjusted for the full confounder set; NB09 candidate pool):')
for ds, pc in partial_corr_results.items():
    n_sig = int((pc['QValue_partial'] < 0.05).sum())
    n_pos = int(((pc['QValue_partial'] < 0.05) & (pc['Rho_partial'] > 0)).sum())
    print(f'  {ds}: {len(pc)} pairs | {n_sig} q<0.05 | {n_pos} positive '
          f'(covariates: {pc["Covariates"].iloc[0]})')

print(f'\n3. Cross-cohort replication (genus level, KEGG-parent matched, '
      f'{analysis_results["consistent_threshold"]}):')
_final_consistent = analysis_results['consistent_pairs']
print(f'  {len(_final_consistent)} genus-polyamine pairs significant in >=2 cohorts')
if not _final_consistent.empty and 'Same_direction' in _final_consistent.columns:
    n_same = int(_final_consistent['Same_direction'].sum())
    print(f'    of which same direction in every cohort tested: {n_same}')
    n_prox = int((_final_consistent['Proxy_used_in'] != 'none').sum())
    print(f'    resting on an N-acetyl proxy in >=1 cohort: {n_prox} '
          f'(weaker evidence; labelled in the CSV)')

print('\n4. Networks (full species-metabolite):')
for ds, G in networks.items():
    print(f'  {ds}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

print('\n5. Stratified hub networks (power-limited by group size -- not a biological trend):')
for group_name in STAGE_GROUP_ORDER:
    G_g = stratified_networks.get(group_name, nx.Graph())
    n_grp = int((harmonized_meta[CRC_DS_HUB]['Stage.Group'] == group_name).sum())
    print(f'  {group_name} (n={n_grp}): {G_g.number_of_nodes()} nodes, '
          f'{G_g.number_of_edges()} edges')

print('\n6. CRC progression 10A (ordinal, FDR fixed inside utils):')
if not prog_corr.empty:
    n_sig_species_u     = prog_corr.loc[prog_corr['QVal_species_stage'] < 0.05, 'Species'].nunique()
    n_sig_metabolites_u = prog_corr.loc[prog_corr['QVal_metabolite_stage'] < 0.05, 'Metabolite'].nunique()
    n_sig_all3 = int(((prog_corr['QVal_species_stage'] < 0.05) &
                      (prog_corr['QVal_metabolite_stage'] < 0.05) &
                      (prog_corr['QVal_spc_mtb'] < 0.05)).sum())
    print(f'  {n_sig_species_u} unique species, {n_sig_metabolites_u} unique metabolites '
          f'track ordinal stage (q<0.05)')
    print(f'  {n_sig_all3} species-metabolite pairs significant in all 3 progression tests')

print('\n7. CRC progression 10B (binary Stage_I_II vs Stage_III_IV):')
if not da_early_late_spc.empty:
    print(f'  {int(da_early_late_spc["Significant"].sum())} species, '
          f'{int(da_early_late_mtb["Significant"].sum())} metabolites significantly different')

print('\n' + '=' * 80)
print(f'Results saved to: {RESULTS_DIR}')
print('  Tables  : Results/tables/*_v2.csv')
print('  Figures : Results/figures/**/*_v2.png')
print('  Pickles : Results/intermediate/preprocessed_data_reduced_v2.pkl, analysis_results_v2.pkl')